## 2.1 Create Working Dataset

The raw dataset `df` is preserved as the source-of-truth dataset and will not be modified directly.

A separate copy, `df_clean`, will be used for all cleaning and transformation operations.

This allows us to:
- preserve the original raw data,
- reproduce the cleaning process,
- compare raw vs cleaned data,
- and debug transformations if necessary.

In [3]:
#import the library
import pandas as pd


#load the dataset as df_clean
df_clean = pd.read_csv("../Data/raw/diabetic_data.csv")


print(F"Dataset shape {df_clean.shape}")

Dataset shape (101766, 50)


## 2.2 Identify Missing-Value Representations

Before handling missing values, we need to determine how missing or unavailable information is represented in the dataset.

We will examine:
- standard Pandas null values,
- the `?` placeholder,
- and other possible placeholder values.

No values will be changed at this stage.

The purpose is to understand the raw representation of missing data before deciding how each variable should be handled.

In [4]:
# Standard missing values
null_counts = df_clean.isna().sum()

# Count '?' placeholders
question_mark_counts = (df_clean == '?').sum()

missing_summary = pd.DataFrame({
    'null_count': null_counts,
    'question_mark_count': question_mark_counts
})

# Show only columns containing missing/placeholder values
missing_summary[
    (missing_summary['null_count'] > 0) |
    (missing_summary['question_mark_count'] > 0)
].sort_values(
    by=['null_count', 'question_mark_count'],
    ascending=False
)

,null_count,question_mark_count
max_glu_serum,96420,0
A1Cresult,84748,0
weight,0,98569
medical_specialty,0,49949
payer_code,0,40256
race,0,2273
diag_3,0,1423
diag_2,0,358
diag_1,0,21


In [5]:
# Check for common textual placeholders
placeholders = ['Unknown', 'unknown', 'UNK', 'N/A', 'NA', 'None']

placeholder_counts = {}

for value in placeholders:
    count = (df_clean == value).sum().sum()
    if count > 0:
        placeholder_counts[value] = count

placeholder_counts

{}

## 2.3 Quantify Missingness

The previous step identified the columns containing missing or placeholder values.

We now calculate the percentage of records affected by missing values.

This helps determine whether a variable has:
- negligible missingness,
- moderate missingness,
- or substantial missingness that may affect its usefulness.

At this stage, we are measuring missingness only. No values will be modified or removed.

In [7]:
# Calculate missing/placeholder percentage
missing_profile = pd.DataFrame({
    'null_count': df_clean.isna().sum(),
    'question_mark_count': (df_clean == '?').sum()
})

missing_profile['missing_count'] = (
    missing_profile['null_count'] +
    missing_profile['question_mark_count']
)

missing_profile['missing_pct'] = (
    missing_profile['missing_count'] / len(df_clean) * 100
)

# Show columns with missing values
missing_profile[
    missing_profile['missing_count'] > 0
].sort_values(
    'missing_pct',
    ascending=False
)

,null_count,question_mark_count,missing_count,missing_pct
weight,0,98569,98569,96.858479
max_glu_serum,96420,0,96420,94.746772
A1Cresult,84748,0,84748,83.277322
medical_specialty,0,49949,49949,49.082208
payer_code,0,40256,40256,39.557416
race,0,2273,2273,2.233555
diag_3,0,1423,1423,1.398306
diag_2,0,358,358,0.351787
diag_1,0,21,21,0.020636


## 2.4 Standardize Missing-Value Representation

The raw dataset uses two representations for missing information:

- `NaN`
- `?`

To make missing-value analysis and subsequent transformations consistent, the `?` placeholder will be converted to Pandas `NaN`.

No rows or columns will be removed, and no values will be imputed at this stage.

The original raw CSV remains unchanged.

In [8]:
# Convert '?' placeholders to Pandas NaN
df_clean = df_clean.replace('?', pd.NA)

# Verify that '?' no longer exists
print("Remaining '?' values:", (df_clean == '?').sum().sum())

# Recalculate missing values
missing_after = df_clean.isna().sum()

missing_after[missing_after > 0].sort_values(ascending=False)

Remaining '?' values: 0


weight               98569
max_glu_serum        96420
A1Cresult            84748
medical_specialty    49949
payer_code           40256
race                  2273
diag_3                1423
diag_2                 358
diag_1                  21
dtype: int64

## 2.5 Missingness and Readmission Outcome

Missingness may contain information about the healthcare encounter.

For example, the absence of a laboratory test may reflect clinical workflow, patient characteristics, or the type of encounter.

Before deciding how to treat missing values, we will compare the readmission distribution between records where a variable is missing and records where it is available.

This analysis is exploratory only. No missing values will be changed at this stage.

In [9]:
# Variables with substantial missingness
missing_columns = [
    'weight',
    'max_glu_serum',
    'A1Cresult',
    'medical_specialty',
    'payer_code'
]

for col in missing_columns:
    print(f"\n{'=' * 60}")
    print(f"{col}")
    print(f"{'=' * 60}")
    
    result = (
        df_clean
        .assign(is_missing=df_clean[col].isna())
        .groupby('is_missing')['readmitted']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .mul(100)
        .round(2)
    )
    
    print(result)


weight
readmitted    <30    >30     NO
is_missing                     
False       11.17  47.23  41.60
True        11.16  34.53  54.31

max_glu_serum
readmitted    <30    >30     NO
is_missing                     
False       12.36  36.21  51.42
True        11.09  34.86  54.05

A1Cresult
readmitted    <30    >30     NO
is_missing                     
False        9.85  34.08  56.07
True        11.42  35.10  53.48

medical_specialty
readmitted    <30    >30     NO
is_missing                     
False       10.76  33.44  55.80
True        11.57  36.47  51.96

payer_code
readmitted    <30    >30     NO
is_missing                     
False       10.94  35.31  53.75
True        11.49  34.35  54.16


## 2.5B Inspect Categories of Variables with Missing Values

Before deciding how missing values should be represented, we need to understand the valid categories of the affected variables.

This is especially important for categorical clinical and administrative variables.

We will inspect:
- the unique categories,
- their frequencies,
- and the proportion represented by missing values.

No values will be modified in this step.

In [11]:
columns_to_inspect = [
    'weight',
    'max_glu_serum',
    'A1Cresult',
    'medical_specialty',
    'payer_code'
]

for col in columns_to_inspect:
    print(f"\n{'=' * 70}")
    print(f"{col}")
    print(f"{'=' * 70}")
    
    print(
        df_clean[col]
        .value_counts(dropna=False)
        .head(20)
    )


weight
weight
NaN          98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

max_glu_serum
max_glu_serum
NaN     96420
Norm     2597
>200     1485
>300     1264
Name: count, dtype: int64

A1Cresult
A1Cresult
NaN     84748
>8       8216
Norm     4990
>7       3812
Name: count, dtype: int64

medical_specialty
medical_specialty
NaN                                  49949
InternalMedicine                     14635
Emergency/Trauma                      7565
Family/GeneralPractice                7440
Cardiology                            5352
Surgery-General                       3099
Nephrology                            1613
Orthopedics                           1400
Orthopedics-Reconstructive            1233
Radiologist                           1140
Pulmonology                            871
Psychiatry                             854
Ur

## 2.6 Handle Missing Categorical Information

For categorical variables where missingness may represent the absence of a recorded test, specialty, or payer, missing values will be represented explicitly as `Not_Documented`.

We will not use mode imputation because assigning the most common category would create information that was not present in the original encounter.

Variables addressed in this step:

- `max_glu_serum`
- `A1Cresult`
- `medical_specialty`
- `payer_code`

The `weight` variable will not be modified yet because its extremely high missingness requires a separate feature-level decision.

In [12]:
categorical_missing_cols = [
    'max_glu_serum',
    'A1Cresult',
    'medical_specialty',
    'payer_code'
]

for col in categorical_missing_cols:
    df_clean[col] = df_clean[col].fillna('Not_Documented')

In [13]:
for col in categorical_missing_cols:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False).head(10))


max_glu_serum
max_glu_serum
Not_Documented    96420
Norm               2597
>200               1485
>300               1264
Name: count, dtype: int64

A1Cresult
A1Cresult
Not_Documented    84748
>8                 8216
Norm               4990
>7                 3812
Name: count, dtype: int64

medical_specialty
medical_specialty
Not_Documented                49949
InternalMedicine              14635
Emergency/Trauma               7565
Family/GeneralPractice         7440
Cardiology                     5352
Surgery-General                3099
Nephrology                     1613
Orthopedics                    1400
Orthopedics-Reconstructive     1233
Radiologist                    1140
Name: count, dtype: int64

payer_code
payer_code
Not_Documented    40256
MC                32439
HM                 6274
SP                 5007
BC                 4655
MD                 3532
CP                 2533
UN                 2448
CM                 1937
OG                 1033
Name: count, dtype: 

## 2.7 Weight Variable — Missingness Decision

The `weight` variable contains 98,569 missing values, representing 96.86% of all encounters.

The observed values are categorical weight ranges rather than continuous measurements.

Because the overwhelming majority of encounters have no recorded weight, imputing a weight category would introduce substantial artificial information into the dataset.

Therefore:

- Missing `weight` values will not be imputed.
- The original observed weight categories will be preserved.
- The variable will remain available for further analysis.
- Its suitability as a machine-learning feature will be evaluated separately during feature selection.

This keeps data cleaning separate from the later modeling decision.

In [14]:
print("Weight missing:", df_clean['weight'].isna().sum())
print("Weight missing %:", round(df_clean['weight'].isna().mean() * 100, 2))

print("\nObserved weight categories:")
print(df_clean['weight'].value_counts(dropna=False))

Weight missing: 98569
Weight missing %: 96.86

Observed weight categories:
weight
NaN          98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64


## 2.8 Investigate Low-Missingness Variables

The variables `race`, `diag_1`, `diag_2`, and `diag_3` contain relatively small proportions of missing values.

Before deciding whether to retain, categorize, or remove these missing records, we will examine their relationship with the readmission outcome.

This helps determine whether the missing observations are randomly distributed or whether missingness may contain information about the encounter.

No values will be modified in this step.

In [15]:
low_missing_cols = [
    'race',
    'diag_1',
    'diag_2',
    'diag_3'
]

for col in low_missing_cols:
    print(f"\n{'=' * 60}")
    print(f"{col}")
    print(f"{'=' * 60}")
    
    missing_target = (
        df_clean
        .assign(is_missing=df_clean[col].isna())
        .groupby('is_missing')['readmitted']
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .mul(100)
        .round(2)
    )
    
    print(missing_target)


race
readmitted    <30    >30     NO
is_missing                     
False       11.23  35.19  53.59
True         8.27  23.67  68.06

diag_1
readmitted    <30    >30     NO
is_missing                     
False       11.16  34.93  53.91
True        23.81  19.05  57.14

diag_2
readmitted    <30    >30     NO
is_missing                     
False       11.17  34.98  53.85
True         8.10  20.67  71.23

diag_3
readmitted    <30    >30     NO
is_missing                     
False       11.23  35.09  53.69
True         6.39  23.75  69.85


In [16]:
for col in low_missing_cols:
    print(
        f"{col}: "
        f"{df_clean[col].isna().sum()} missing "
        f"({df_clean[col].isna().mean() * 100:.2f}%)"
    )

race: 2273 missing (2.23%)
diag_1: 21 missing (0.02%)
diag_2: 358 missing (0.35%)
diag_3: 1423 missing (1.40%)


## 2.8 Handle Low-Missingness Categorical Values

The variables `race`, `diag_1`, `diag_2`, and `diag_3` contain relatively low levels of missing information.

Rather than deleting the affected encounters or assigning an existing category, missing values will be explicitly represented as `Not_Documented`.

This preserves all encounters and prevents the cleaning process from creating information that was not present in the original data.

The diagnosis variables will be examined further during feature engineering because their clinical codes require additional transformation before machine learning.

In [17]:
low_missing_cols = [
    'race',
    'diag_1',
    'diag_2',
    'diag_3'
]

for col in low_missing_cols:
    df_clean[col] = df_clean[col].fillna('Not_Documented')

In [18]:
for col in low_missing_cols:
    print(f"\n{col}")
    print(df_clean[col].value_counts(dropna=False).head(10))


race
race
Caucasian          76099
AfricanAmerican    19210
Not_Documented      2273
Hispanic            2037
Other               1506
Asian                641
Name: count, dtype: int64

diag_1
diag_1
428    6862
414    6581
786    4016
410    3614
486    3508
427    2766
491    2275
715    2151
682    2042
434    2028
Name: count, dtype: int64

diag_2
diag_2
276    6752
428    6662
250    6071
427    5036
401    3736
496    3305
599    3288
403    2823
414    2650
411    2566
Name: count, dtype: int64

diag_3
diag_3
250    11555
401     8289
276     5175
428     4577
427     3955
414     3664
496     2605
403     2357
585     1992
272     1969
Name: count, dtype: int64


In [19]:
for col in ['race', 'diag_1', 'diag_2', 'diag_3']:
    print(
        f"{col}:",
        df_clean[col].isna().sum(),
        "NaN remaining |",
        (df_clean[col] == 'Not_Documented').sum(),
        "Not_Documented"
    )

race: 0 NaN remaining | 2273 Not_Documented
diag_1: 0 NaN remaining | 21 Not_Documented
diag_2: 0 NaN remaining | 358 Not_Documented
diag_3: 0 NaN remaining | 1423 Not_Documented


## 2.9 Data Type Audit

After standardizing missing values, the next step is to audit the data types of all variables.

A variable's storage type should reflect its meaning.

For example:
- numeric measurements should remain numeric,
- categorical variables should be represented as categorical/text data,
- identifiers and coded variables should not automatically be treated as continuous numerical measurements.

No data types will be changed during this audit. We will first inspect the current structure and identify variables that require transformation.

In [20]:
# Inspect current data types
dtype_summary = pd.DataFrame({
    'dtype': df_clean.dtypes,
    'unique_values': df_clean.nunique(dropna=False),
    'missing_values': df_clean.isna().sum()
})

dtype_summary

,dtype,unique_values,missing_values
encounter_id,int64,101766,0
patient_nbr,int64,71518,0
race,str,6,0
gender,str,3,0
age,str,10,0
weight,str,10,98569
admission_type_id,int64,8,0
discharge_disposition_id,int64,26,0
admission_source_id,int64,17,0
time_in_hospital,int64,14,0


In [21]:
# Display columns grouped by their current pandas dtype
for dtype in df_clean.dtypes.unique():
    print(f"\n{'=' * 60}")
    print(f"Dtype: {dtype}")
    print(f"{'=' * 60}")
    print(df_clean.select_dtypes(include=[dtype]).columns.tolist())


Dtype: int64
['encounter_id', 'patient_nbr', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses']

Dtype: str
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'readmitted']


## 2.10 Audit Coded Categorical Variables

Several variables are stored as integers but represent categorical codes rather than continuous numerical measurements.

The main coded variables are:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

Before changing their data types, we will inspect their unique values and frequency distributions.

This is important because the codes themselves do not have numerical meaning, and some discharge dispositions may have special implications for the readmission prediction problem.

No data types will be changed in this step.

In [23]:
coded_cols = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

for col in coded_cols:
    print(f"\n{'=' * 70}")
    print(col)
    print(f"{'=' * 70}")
    
    print("Unique values:")
    print(sorted(df_clean[col].unique()))
    
    print("\nFrequency:")
    print(df_clean[col].value_counts().sort_index())


admission_type_id
Unique values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]

Frequency:
admission_type_id
1    53990
2    18480
3    18869
4       10
5     4785
6     5291
7       21
8      320
Name: count, dtype: int64

discharge_disposition_id
Unique values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(27), np.int64(28)]

Frequency:
discharge_disposition_id
1     60234
2      2128
3     13954
4       815
5      1184
6     12902
7       623
8       108
9        21
10        6
11     1642
12        3
13      399
14      372
15       63
16       11
17       14
18     3691
19        8
20        2
22     1993
23      412
24       48
25     

## 2.11 Convert Coded Variables to Categorical Type

The audit confirmed that `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` are categorical codes rather than continuous numerical measurements.

They will therefore be converted to Pandas categorical dtype.

The category values themselves will not be changed or grouped at this stage.

Rare categories will be preserved because frequency alone does not establish that a category is invalid.

In [25]:
coded_cols = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

for col in coded_cols:
    df_clean[col] = df_clean[col].astype('category')

In [26]:
print(df_clean[coded_cols].dtypes)

print("\nNumber of categories:")
for col in coded_cols:
    print(f"{col}: {df_clean[col].nunique()}")

admission_type_id           category
discharge_disposition_id    category
admission_source_id         category
dtype: object

Number of categories:
admission_type_id: 8
discharge_disposition_id: 26
admission_source_id: 17


## 2.12 Audit Remaining Categorical Variables

The dataset contains several string-based variables representing different types of information.

Before converting their data types, we will inspect their unique values and frequencies.

This allows us to distinguish:
- nominal categorical variables,
- ordinal variables,
- binary variables,
- medication-status variables,
- diagnosis codes,
- and the target variable.

No additional transformations will be performed until the structure of these variables is understood.

In [32]:
categorical_cols = [
    'gender',
    'age',
    'weight',
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'acetohexamide',
    'glipizide',
    'glyburide',
    'tolbutamide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'examide',
    'citoglipton',
    'insulin',
    'glyburide-metformin',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone',
    'change',
    'diabetesMed',
    'readmitted'
]

for col in categorical_cols:
    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")
    print(df_clean[col].value_counts(dropna=False))


gender
gender
Female             54708
Male               47055
Unknown/Invalid        3
Name: count, dtype: int64

age
age
[70-80)     26068
[60-70)     22483
[50-60)     17256
[80-90)     17197
[40-50)      9685
[30-40)      3775
[90-100)     2793
[20-30)      1657
[10-20)       691
[0-10)        161
Name: count, dtype: int64

weight
weight
NaN          98569
[75-100)      1336
[50-75)        897
[100-125)      625
[125-150)      145
[25-50)         97
[0-25)          48
[150-175)       35
[175-200)       11
>200             3
Name: count, dtype: int64

metformin
metformin
No        81778
Steady    18346
Up         1067
Down        575
Name: count, dtype: int64

repaglinide
repaglinide
No        100227
Steady      1384
Up           110
Down          45
Name: count, dtype: int64

nateglinide
nateglinide
No        101063
Steady       668
Up            24
Down          11
Name: count, dtype: int64

chlorpropamide
chlorpropamide
No        101680
Steady        79
Up             6
Down   

## 2.13 Handle Invalid Gender Values

The `gender` variable contains two valid categories:

- `Female`
- `Male`

and three records labeled `Unknown/Invalid`.

Because these three records do not represent a valid gender category and the number of affected encounters is negligible, they will be represented as `Not_Documented`.

The encounters themselves will not be removed.

In [33]:
# Replace invalid gender values with an explicit missing category
df_clean['gender'] = df_clean['gender'].replace(
    'Unknown/Invalid',
    'Not_Documented'
)

# Validate
print(df_clean['gender'].value_counts(dropna=False))

gender
Female            54708
Male              47055
Not_Documented        3
Name: count, dtype: int64


## 2.14 Age-Band Validation

The `age` variable consists of ten ordered age bands ranging from `[0-10)` to `[90-100)`.

Because these categories have an inherent order, they should not be treated as arbitrary nominal labels.

Before defining the final representation, we will validate that all expected age bands are present and that no unexpected values exist.

No values will be transformed during this validation.

In [34]:
expected_age_bands = [
    '[0-10)',
    '[10-20)',
    '[20-30)',
    '[30-40)',
    '[40-50)',
    '[50-60)',
    '[60-70)',
    '[70-80)',
    '[80-90)',
    '[90-100)'
]

actual_age_bands = df_clean['age'].unique().tolist()

print("Actual age bands:")
print(sorted(actual_age_bands))

print("\nUnexpected values:")
print(set(actual_age_bands) - set(expected_age_bands))

print("\nMissing expected bands:")
print(set(expected_age_bands) - set(actual_age_bands))

Actual age bands:
['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)', '[60-70)', '[70-80)', '[80-90)', '[90-100)']

Unexpected values:
set()

Missing expected bands:
set()


## 2.14B Convert Age to Ordered Categorical Type

The `age` variable contains ten validated age bands with a natural chronological order.

The variable will be converted to an ordered categorical type so that the ordering of the age bands is explicitly represented.

The original age-band labels will be preserved.

No midpoint or exact age will be inferred from the ranges.

In [35]:
age_order = [
    '[0-10)',
    '[10-20)',
    '[20-30)',
    '[30-40)',
    '[40-50)',
    '[50-60)',
    '[60-70)',
    '[70-80)',
    '[80-90)',
    '[90-100)'
]

df_clean['age'] = pd.Categorical(
    df_clean['age'],
    categories=age_order,
    ordered=True
)

In [36]:
print(df_clean['age'].dtype)

print("\nAge categories:")
print(df_clean['age'].cat.categories)

print("\nOrdered:")
print(df_clean['age'].cat.ordered)

category

Age categories:
Index(['[0-10)', '[10-20)', '[20-30)', '[30-40)', '[40-50)', '[50-60)',
       '[60-70)', '[70-80)', '[80-90)', '[90-100)'],
      dtype='str')

Ordered:
True


## 2.15 Medication Variable Audit

The medication variables describe whether a medication was not prescribed, continued at the same level, increased, or decreased.

We will examine the number of unique values and the proportion represented by the most common category.

This identifies:
- constant variables with no variation,
- highly imbalanced variables,
- and medication variables with meaningful variation.

A variable being highly imbalanced does not automatically make it invalid or unsuitable for modeling, so no columns will be removed at this stage.

In [37]:
medication_cols = [
    'metformin',
    'repaglinide',
    'nateglinide',
    'chlorpropamide',
    'glimepiride',
    'acetohexamide',
    'glipizide',
    'glyburide',
    'tolbutamide',
    'pioglitazone',
    'rosiglitazone',
    'acarbose',
    'miglitol',
    'troglitazone',
    'tolazamide',
    'examide',
    'citoglipton',
    'insulin',
    'glyburide-metformin',
    'glipizide-metformin',
    'glimepiride-pioglitazone',
    'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

medication_audit = pd.DataFrame({
    'unique_values': df_clean[medication_cols].nunique(),
    'most_common_count': [
        df_clean[col].value_counts().iloc[0]
        for col in medication_cols
    ]
})

medication_audit['most_common_pct'] = (
    medication_audit['most_common_count']
    / len(df_clean)
    * 100
)

medication_audit.sort_values(
    'most_common_pct',
    ascending=False
)

,unique_values,most_common_count,most_common_pct
citoglipton,1,101766,100.000000
examide,1,101766,100.000000
acetohexamide,2,101765,99.999017
metformin-pioglitazone,2,101765,99.999017
glimepiride-pioglitazone,2,101765,99.999017
metformin-rosiglitazone,2,101764,99.998035
troglitazone,2,101763,99.997052
glipizide-metformin,2,101753,99.987226
tolbutamide,2,101743,99.977399
miglitol,4,101728,99.962659


## 2.15B Remove Constant Medication Variables

The medication audit identified two variables with no variation:

- `citoglipton` — 100% `No`
- `examide` — 100% `No`

Because these variables contain a single value for every encounter, they provide no analytical or predictive information.

They will be removed from the cleaned working dataset.

Near-constant medication variables will be retained for now because rarity alone does not establish that a variable should be removed. Their usefulness will be evaluated during later feature selection.


In [40]:
constant_medication_cols = [
    'citoglipton',
    'examide'
]

df_clean = df_clean.drop(columns=constant_medication_cols)

In [41]:
print("Removed columns:", constant_medication_cols)

print("\nColumns still present:")
print(
    [col for col in constant_medication_cols if col in df_clean.columns]
)

print("\nCurrent dataset shape:")
print(df_clean.shape)

Removed columns: ['citoglipton', 'examide']

Columns still present:
[]

Current dataset shape:
(101766, 48)


## 2.16 Diagnosis Code Audit

The variables `diag_1`, `diag_2`, and `diag_3` contain diagnosis codes.

Although many values appear numeric, these are clinical classification codes rather than continuous measurements.

Before transforming them, we will inspect:
- the data types,
- representative values,
- codes containing decimals,
- codes beginning with `V` or `E`,
- and the proportion of records represented by each format.

This is necessary before deciding how diagnosis information should be cleaned and transformed for later analysis and machine learning.

In [42]:
diagnosis_cols = ['diag_1', 'diag_2', 'diag_3']

for col in diagnosis_cols:
    print(f"\n{'=' * 70}")
    print(col)
    print(f"{'=' * 70}")
    
    print("Data type:", df_clean[col].dtype)
    print("Unique values:", df_clean[col].nunique())
    
    print("\nSample values:")
    print(df_clean[col].drop_duplicates().head(30).tolist())


diag_1
Data type: str
Unique values: 717

Sample values:
['250.83', '276', '648', '8', '197', '414', '428', '398', '434', '250.7', '157', '518', '999', '410', '682', '402', '737', '572', 'V57', '189', '786', '427', '996', '277', '584', '462', '473', '411', '174', '486']

diag_2
Data type: str
Unique values: 749

Sample values:
['Not_Documented', '250.01', '250', '250.43', '157', '411', '492', '427', '198', '403', '288', '998', '507', '174', '425', '456', '401', '715', '496', '428', '585', '250.02', '410', '999', '996', '135', '244', '41', '571', '276']

diag_3
Data type: str
Unique values: 790

Sample values:
['Not_Documented', '255', 'V27', '403', '250', 'V45', '38', '486', '996', '197', '250.6', '427', '627', '414', '416', '714', '428', '582', 'V43', '250.01', '263', '250.42', '276', '482', '401', '250.41', '585', '781', '278', '998']


In [43]:
for col in diagnosis_cols:
    print(f"\n{col}")
    
    values = df_clean[col].dropna().astype(str)
    
    print("Contains decimal:", values.str.contains(r'\.', regex=True).sum())
    print("Starts with V:", values.str.startswith('V').sum())
    print("Starts with E:", values.str.startswith('E').sum())


diag_1
Contains decimal: 8522
Starts with V: 1644
Starts with E: 1

diag_2
Contains decimal: 6723
Starts with V: 1805
Starts with E: 731

diag_3
Contains decimal: 5603
Starts with V: 3814
Starts with E: 1244


## 2.16B Validate Diagnosis Code Format

The diagnosis variables contain several valid ICD-9-style formats:

- numeric codes,
- decimal codes,
- V-codes,
- E-codes,
- and `Not_Documented` for unavailable diagnoses.

The diagnosis values will remain strings because their numeric-looking values are classification codes rather than continuous measurements.

We will now identify values that do not match the expected structural patterns.

No diagnosis values will be modified during this validation.

In [44]:
import re

diagnosis_pattern = r'^(?:\d{1,3}(?:\.\d{1,2})?|[VE]\d{2,3})$'

for col in diagnosis_cols:
    values = df_clean[col].dropna().astype(str)

    invalid_values = values[
        ~values.str.match(diagnosis_pattern)
        & (values != 'Not_Documented')
    ]

    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")
    
    print("Invalid-format records:", len(invalid_values))
    
    if len(invalid_values) > 0:
        print("\nInvalid values:")
        print(invalid_values.value_counts())


diag_1
Invalid-format records: 0

diag_2
Invalid-format records: 0

diag_3
Invalid-format records: 0


## 2.17 Duplicate Record Audit

Before continuing with data transformation, we need to determine whether duplicate encounters or duplicate rows exist.

The `encounter_id` should uniquely identify each hospital encounter.

Repeated `patient_nbr` values are expected because the same patient can have multiple hospital encounters and should not automatically be treated as duplicates.

We will therefore check:

- duplicate `encounter_id` values,
- completely duplicated rows,
- and the number of unique patients.

No records will be removed during this audit.

In [45]:
print("Total rows:", len(df_clean))

print("\nUnique encounter IDs:", df_clean['encounter_id'].nunique())
print("Duplicate encounter IDs:", df_clean['encounter_id'].duplicated().sum())

print("\nCompletely duplicated rows:", df_clean.duplicated().sum())

print("\nUnique patients:", df_clean['patient_nbr'].nunique())
print("Patients with multiple encounters:",
      (df_clean['patient_nbr'].value_counts() > 1).sum())

Total rows: 101766

Unique encounter IDs: 101766
Duplicate encounter IDs: 0

Completely duplicated rows: 0

Unique patients: 71518
Patients with multiple encounters: 16773


## 2.18 Target Variable Audit

The raw target variable is `readmitted`, containing three outcome categories:

- `<30` — readmission within 30 days
- `>30` — readmission after 30 days
- `NO` — no readmission

The final machine-learning problem will convert this into a binary target where `<30` is the positive class.

Before performing that transformation, we will validate the target structure and class distribution.

The raw target will not be modified during this audit.

In [46]:
print("Target dtype:", df_clean['readmitted'].dtype)

print("\nMissing target values:")
print(df_clean['readmitted'].isna().sum())

print("\nUnique target values:")
print(df_clean['readmitted'].unique())

print("\nTarget distribution:")
print(df_clean['readmitted'].value_counts())

print("\nTarget distribution (%):")
print(
    df_clean['readmitted']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Target dtype: str

Missing target values:
0

Unique target values:
<StringArray>
['NO', '>30', '<30']
Length: 3, dtype: str

Target distribution:
readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Target distribution (%):
readmitted
NO     53.91
>30    34.93
<30    11.16
Name: proportion, dtype: float64


## 2.19 Create Binary 30-Day Readmission Target

The raw `readmitted` variable contains three outcome categories:

- `<30` — readmitted within 30 days
- `>30` — readmitted after 30 days
- `NO` — not readmitted

The project's machine-learning objective is specifically to predict readmission within 30 days.

Therefore, a separate binary target `readmitted_30d` will be created:

- `<30` → 1 (positive class)
- `>30` → 0
- `NO` → 0

The original `readmitted` variable will be preserved for traceability and validation.

In [47]:
# Create binary 30-day readmission target
df_clean['readmitted_30d'] = ( df_clean['readmitted'] == '<30').astype(int)

In [ ]:
print("Target mapping:")
print(df_clean.groupby('readmitted')['readmitted_30d'].value_counts())

print("\nBinary target distribution:")
print(df_clean['readmitted_30d'].value_counts())

print("\nBinary target distribution (%):")
print(df_clean['readmitted_30d'].value_counts(normalize=True).mul(100).round(2))

Target mapping:
readmitted  readmitted_30d
<30         1                 11357
>30         0                 35545
NO          0                 54864
Name: count, dtype: int64

Binary target distribution:
readmitted_30d
0    90409
1    11357
Name: count, dtype: int64

Binary target distribution (%):
readmitted_30d
0    88.84
1    11.16
Name: proportion, dtype: float64


## 2.20 Numeric Variable Validation

The numerical variables will be checked for structurally invalid values.

The objective is to identify:
- negative counts,
- impossible zero/negative values where they should not occur,
- unexpected ranges,
- and other obvious data-quality problems.

This is a validation step only.

No values will be changed until an invalid or suspicious pattern is identified and interpreted.


In [50]:
numeric_cols = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'number_diagnoses'
]

numeric_audit = pd.DataFrame({
    'min': df_clean[numeric_cols].min(),
    'max': df_clean[numeric_cols].max(),
    'zero_count': (df_clean[numeric_cols] == 0).sum(),
    'negative_count': (df_clean[numeric_cols] < 0).sum(),
    'unique_values': df_clean[numeric_cols].nunique()
})

numeric_audit

,min,max,zero_count,negative_count,unique_values
time_in_hospital,1,14,0,0,14
num_lab_procedures,1,132,0,0,118
num_procedures,0,6,46652,0,7
num_medications,1,81,0,0,75
number_outpatient,0,42,85027,0,39
number_emergency,0,76,90383,0,33
number_inpatient,0,21,67630,0,21
number_diagnoses,1,16,0,0,16


In [51]:
print("Negative values by column:")
print(
    (df_clean[numeric_cols] < 0).sum()
)

print("\nMissing values by column:")
print(
    df_clean[numeric_cols].isna().sum()
)


Negative values by column:
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
num_medications       0
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      0
dtype: int64

Missing values by column:
time_in_hospital      0
num_lab_procedures    0
num_procedures        0
num_medications       0
number_outpatient     0
number_emergency      0
number_inpatient      0
number_diagnoses      0
dtype: int64


## 2.21 Discharge Disposition and Target Eligibility Audit

The prediction objective is to estimate the risk of readmission within 30 days at or near hospital discharge.

Some discharge dispositions may represent outcomes where subsequent readmission is impossible or not comparable to standard discharge pathways.

Before making any exclusion decision, we will examine the relationship between `discharge_disposition_id` and the readmission outcome.

No encounters will be removed during this audit.

In [52]:
discharge_target = pd.crosstab(
    df_clean['discharge_disposition_id'],
    df_clean['readmitted'],
    margins=True
)

discharge_target

readmitted,<30,>30,NO,All
discharge_disposition_id,,,,
1,5602,21517,33115,60234
2,342,668,1118,2128
3,2046,4916,6992,13954
4,104,278,433,815
5,247,350,587,1184
6,1638,5362,5902,12902
7,90,221,312,623
8,15,38,55,108
9,9,2,10,21


In [53]:
# Identify discharge codes with no <30 readmissions
zero_30day = discharge_target[
    (discharge_target['<30'] == 0) &
    (discharge_target.index != 'All')
]

zero_30day

readmitted,<30,>30,NO,All
discharge_disposition_id,,,,
10,0,4,2,6
11,0,0,1642,1642
16,0,6,5,11
17,0,5,9,14
19,0,0,8,8
20,0,0,2,2
27,0,1,4,5


## 2.21B Load Administrative ID Mapping

The dataset includes an `IDs_mapping.csv` file containing descriptions for the coded administrative variables.

Rather than manually assigning meanings to numeric IDs, we will use the supplied mapping to decode:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

This allows us to distinguish legitimate categories from special dispositions that may affect cohort eligibility.

No records will be removed at this stage.

## 2.21C Inspect Official Discharge Disposition Mapping

The official `IDs_mapping.csv` file will be used to decode the administrative IDs in the raw dataset.

We will first inspect the mapping for `discharge_disposition_id` because discharge disposition is relevant to the eligibility of encounters for a 30-day readmission prediction problem.

No values in `df_clean` will be changed at this stage.

In [55]:
mapping = pd.read_csv("../data/raw/IDs_mapping.csv")

print("Mapping shape:", mapping.shape)
print("\nColumns:")
print(mapping.columns.tolist())

print("\nFirst rows:")
display(mapping.head(20))

Mapping shape: (67, 2)

Columns:
['admission_type_id', 'description']

First rows:


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


In [56]:
print(mapping.to_string(index=False))


       admission_type_id                                                                                               description
                       1                                                                                                 Emergency
                       2                                                                                                    Urgent
                       3                                                                                                  Elective
                       4                                                                                                   Newborn
                       5                                                                                             Not Available
                       6                                                                                                       NaN
                       7                                                           

## 2.21D Special Discharge Dispositions

The official `IDs_mapping.csv` mapping was used to interpret `discharge_disposition_id`.

Several codes represent special discharge outcomes, including death and hospice:

- 11 — Expired
- 13 — Hospice / home
- 14 — Hospice / medical facility
- 19 — Expired at home / Medicaid-only hospice
- 20 — Expired in a medical facility / Medicaid-only hospice
- 21 — Expired, place unknown

These categories require special consideration for the 30-day readmission prediction cohort.

They will not be removed during general data cleaning.

Instead, cohort eligibility will be handled explicitly during construction of the leakage-controlled modeling dataset.

This separation prevents general cleaning decisions from being confused with the clinical definition of the prediction population.

In [ ]:
special_discharge_ids = [11, 13, 14, 19, 20, 21]

special_discharge = (
    df_clean[
        df_clean['discharge_disposition_id']
        .isin(special_discharge_ids)
    ]
    ['discharge_disposition_id']
    .value_counts()
    .sort_index()
)

special_discharge

discharge_disposition_id
1        0
2        0
3        0
4        0
5        0
6        0
7        0
8        0
9        0
10       0
11    1642
12       0
13     399
14     372
15       0
16       0
17       0
18       0
19       8
20       2
22       0
23       0
24       0
25       0
27       0
28       0
Name: count, dtype: int64

In [58]:
print("Total special-disposition encounters:", special_discharge.sum())

print(
    "Percentage of all encounters:",
    round(
        special_discharge.sum() / len(df_clean) * 100,
        2
    )
)

Total special-disposition encounters: 2423
Percentage of all encounters: 2.38


## 2.22 Final Categorical Consistency Audit

The major categorical variables have been individually examined during the cleaning process.

A final consistency audit will summarize the remaining categorical variables to identify unexpected missing values, unusually small category counts, or variables that may require further investigation.

This is a validation step only. No values will be changed or removed.

In [60]:
categorical_like_cols = df_clean.select_dtypes(
    include=['str', 'category']
).columns

categorical_summary = pd.DataFrame({
    'dtype': df_clean[categorical_like_cols].dtypes,
    'unique_values': df_clean[categorical_like_cols].nunique(
        dropna=False
    ),
    'missing_values': df_clean[categorical_like_cols].isna().sum(),
    'most_common_value': [
        df_clean[col].value_counts(dropna=False).index[0]
        for col in categorical_like_cols
    ],
    'most_common_count': [
        df_clean[col].value_counts(dropna=False).iloc[0]
        for col in categorical_like_cols
    ]
})

categorical_summary['most_common_pct'] = (
    categorical_summary['most_common_count']
    / len(df_clean)
    * 100
).round(2)

categorical_summary

,dtype,unique_values,missing_values,most_common_value,most_common_count,most_common_pct
race,str,6,0,Caucasian,76099,74.78
gender,str,3,0,Female,54708,53.76
age,category,10,0,[70-80),26068,25.62
weight,str,10,98569,NaN,98569,96.86
admission_type_id,category,8,0,1,53990,53.05
discharge_disposition_id,category,26,0,1,60234,59.19
admission_source_id,category,17,0,7,57494,56.50
payer_code,str,18,0,Not_Documented,40256,39.56
medical_specialty,str,73,0,Not_Documented,49949,49.08
diag_1,str,717,0,428,6862,6.74


In [61]:
print("Categorical columns with remaining NaN:")
print(
    categorical_summary[
        categorical_summary['missing_values'] > 0
    ]
)

Categorical columns with remaining NaN:
       dtype  unique_values  missing_values most_common_value  \
weight   str             10           98569               NaN   

        most_common_count  most_common_pct  
weight              98569            96.86  


## 2.23 Cleaning Decisions and Transformation Log

The cleaning process was performed incrementally while preserving the raw CSV as the source of truth.

### Changes applied

1. Converted `?` placeholders to missing values.
2. Represented missing categorical information as `Not_Documented` where appropriate.
3. Preserved `weight` missingness because 96.86% of values are unavailable; no imputation was performed.
4. Converted `admission_type_id`, `discharge_disposition_id`, and `admission_source_id` from integer codes to categorical dtype.
5. Converted `age` into an ordered categorical variable while preserving the original age bands.
6. Replaced three `Unknown/Invalid` gender values with `Not_Documented`.
7. Removed two constant medication variables: `examide` and `citoglipton`.
8. Preserved diagnosis codes as strings after validating their ICD-9-style structure.
9. Verified that there are no duplicate encounter IDs or completely duplicated rows.
10. Validated numerical variables and found no negative or missing values.
11. Preserved the original three-class `readmitted` outcome.
12. Created `readmitted_30d` as the binary modeling target.
13. Documented special discharge dispositions for later cohort-eligibility decisions.

No rows have been removed during the cleaning process.
The original raw CSV remains unchanged.

## 2.24A Create Human-Readable Administrative Variables

The dataset contains three administrative variables represented by numeric IDs.

The official `IDs_mapping.csv` file provides descriptions for these IDs.

To improve interpretability, human-readable categorical variables will be created while preserving the original ID columns.

Original ID columns will not be overwritten.

In [62]:
# Inspect the mapping structure
mapping.head(15)

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


In [63]:
# Identify section boundaries in the mapping file
section_starts = mapping[
    mapping['admission_type_id'].isin([
        'admission_type_id',
        'discharge_disposition_id',
        'admission_source_id'
    ])
]

section_starts

,admission_type_id,description
9,discharge_disposition_id,description
41,admission_source_id,description


In [65]:
# Convert the first mapping column to string for safe comparison
mapping_key = mapping['admission_type_id'].astype(str).str.strip()

section_starts = mapping[
    mapping_key.isin([
        'admission_type_id',
        'discharge_disposition_id',
        'admission_source_id'
    ])
]

section_starts

,admission_type_id,description
9,discharge_disposition_id,description
41,admission_source_id,description


In [66]:
for i, row in mapping.iterrows():
    value = str(row['admission_type_id']).strip()
    
    if value in [
        'admission_type_id',
        'discharge_disposition_id',
        'admission_source_id'
    ]:
        print(i, value)

9 discharge_disposition_id
41 admission_source_id


## 2.24C Create Administrative Lookup Mappings

The official mapping file contains three separate sections.

Lookup dictionaries will be created for:
- admission type,
- discharge disposition,
- admission source.

The original ID variables will be preserved.

Where the official mapping contains no description for an existing ID, the description will be represented as `Not_Documented` rather than introducing a new missing value.

In [67]:
# Create separate sections from the official mapping file

admission_mapping = mapping.iloc[0:8].copy()

discharge_mapping = mapping.iloc[10:40].copy()

source_mapping = mapping.iloc[42:].copy()

# Convert ID columns to numeric
admission_mapping['admission_type_id'] = pd.to_numeric(
    admission_mapping['admission_type_id'],
    errors='coerce'
)

discharge_mapping['admission_type_id'] = pd.to_numeric(
    discharge_mapping['admission_type_id'],
    errors='coerce'
)

source_mapping['admission_type_id'] = pd.to_numeric(
    source_mapping['admission_type_id'],
    errors='coerce'
)

# Remove rows without an ID
admission_mapping = admission_mapping.dropna(
    subset=['admission_type_id']
)

discharge_mapping = discharge_mapping.dropna(
    subset=['admission_type_id']
)

source_mapping = source_mapping.dropna(
    subset=['admission_type_id']
)

# Replace missing descriptions with explicit label
for table in [
    admission_mapping,
    discharge_mapping,
    source_mapping
]:
    table['description'] = table['description'].fillna(
        'Not_Documented'
    )

In [68]:
admission_mapping = admission_mapping.rename(
    columns={'admission_type_id': 'id'}
)

discharge_mapping = discharge_mapping.rename(
    columns={'admission_type_id': 'id'}
)

source_mapping = source_mapping.rename(
    columns={'admission_type_id': 'id'}
)

admission_map = dict(
    zip(admission_mapping['id'], admission_mapping['description'])
)

discharge_map = dict(
    zip(discharge_mapping['id'], discharge_mapping['description'])
)

source_map = dict(
    zip(source_mapping['id'], source_mapping['description'])
)

In [69]:
print("Admission mapping:")
print(admission_map)

print("\nDischarge mapping:")
print(discharge_map)

print("\nSource mapping:")
print(source_map)

Admission mapping:
{1: 'Emergency', 2: 'Urgent', 3: 'Elective', 4: 'Newborn', 5: 'Not Available', 6: 'Not_Documented', 7: 'Trauma Center', 8: 'Not Mapped'}

Discharge mapping:
{1: 'Discharged to home', 2: 'Discharged/transferred to another short term hospital', 3: 'Discharged/transferred to SNF', 4: 'Discharged/transferred to ICF', 5: 'Discharged/transferred to another type of inpatient care institution', 6: 'Discharged/transferred to home with home health service', 7: 'Left AMA', 8: 'Discharged/transferred to home under care of Home IV provider', 9: 'Admitted as an inpatient to this hospital', 10: 'Neonate discharged to another hospital for neonatal aftercare', 11: 'Expired', 12: 'Still patient or expected to return for outpatient services', 13: 'Hospice / home', 14: 'Hospice / medical facility', 15: 'Discharged/transferred within this institution to Medicare approved swing bed', 16: 'Discharged/transferred/referred another institution for outpatient services', 17: 'Discharged/transfe

## 2.24D Apply Administrative Code Transformations

The validated official ID mappings will be applied to the cleaned dataset.

Three human-readable variables will be created:

- `admission_type`
- `discharge_disposition`
- `admission_source`

The original ID variables will be retained because they preserve the source representation and may be useful for reproducibility or later analysis.

Mapping descriptions will also be stripped of unnecessary leading/trailing whitespace.

In [70]:
# Remove unnecessary whitespace from mapping descriptions
admission_map = {
    key: value.strip()
    for key, value in admission_map.items()
}

discharge_map = {
    key: value.strip()
    for key, value in discharge_map.items()
}

source_map = {
    key: value.strip()
    for key, value in source_map.items()
}

In [71]:
df_clean['admission_type'] = (
    df_clean['admission_type_id']
    .map(admission_map)
)

df_clean['discharge_disposition'] = (
    df_clean['discharge_disposition_id']
    .map(discharge_map)
)

df_clean['admission_source'] = (
    df_clean['admission_source_id']
    .map(source_map)
)

In [72]:
mapping_validation = pd.DataFrame({
    'source_column': [
        'admission_type_id',
        'discharge_disposition_id',
        'admission_source_id'
    ],
    'transformed_column': [
        'admission_type',
        'discharge_disposition',
        'admission_source'
    ],
    'unmapped_values': [
        df_clean['admission_type'].isna().sum(),
        df_clean['discharge_disposition'].isna().sum(),
        df_clean['admission_source'].isna().sum()
    ]
})

mapping_validation

,source_column,transformed_column,unmapped_values
0,admission_type_id,admission_type,0
1,discharge_disposition_id,discharge_disposition,0
2,admission_source_id,admission_source,0


In [73]:
print(df_clean[
    [
        'admission_type_id',
        'admission_type',
        'discharge_disposition_id',
        'discharge_disposition',
        'admission_source_id',
        'admission_source'
    ]
].head(10))

  admission_type_id  admission_type discharge_disposition_id  \
0                 6  Not_Documented                       25   
1                 1       Emergency                        1   
2                 1       Emergency                        1   
3                 1       Emergency                        1   
4                 1       Emergency                        1   
5                 2          Urgent                        1   
6                 3        Elective                        1   
7                 1       Emergency                        1   
8                 2          Urgent                        1   
9                 3        Elective                        3   

           discharge_disposition admission_source_id          admission_source  
0                     Not Mapped                   1        Physician Referral  
1             Discharged to home                   7            Emergency Room  
2             Discharged to home                   7

In [74]:
mapping_validation

,source_column,transformed_column,unmapped_values
0,admission_type_id,admission_type,0
1,discharge_disposition_id,discharge_disposition,0
2,admission_source_id,admission_source,0


## 2.25A Diagnosis Code Structure

The diagnosis variables contain ICD-9 diagnosis codes.

For analytical purposes, the detailed codes can later be grouped into broader clinical categories.

Before creating those categories, we will examine the structure of the codes:

- numeric ICD-9 codes,
- V-codes,
- E-codes,
- and `Not_Documented`.

The original diagnosis codes will be preserved.

In [75]:
for col in diagnosis_cols:
    values = df_clean[col].astype(str)

    structure = pd.Series(
        'Numeric',
        index=values.index
    )

    structure[values.str.startswith('V')] = 'V-code'
    structure[values.str.startswith('E')] = 'E-code'
    structure[values == 'Not_Documented'] = 'Not_Documented'

    print(f"\n{'=' * 60}")
    print(col)
    print(f"{'=' * 60}")
    print(structure.value_counts())


diag_1
Numeric           100100
V-code              1644
Not_Documented        21
E-code                 1
Name: count, dtype: int64

diag_2
Numeric           98872
V-code             1805
E-code              731
Not_Documented      358
Name: count, dtype: int64

diag_3
Numeric           95285
V-code             3814
Not_Documented     1423
E-code             1244
Name: count, dtype: int64


## 2.25B Create Broad Diagnosis Groups

The detailed ICD-9 diagnosis codes will be transformed into broader clinical groups using ICD-9 chapter ranges.

The original diagnosis codes will be preserved.

Numeric ICD-9 codes will be assigned to their corresponding clinical chapter.

V-codes and E-codes will be retained as separate groups.

`Not_Documented` will remain explicitly represented as `Not_Documented`.

Three new variables will be created:

- `diag_1_group`
- `diag_2_group`
- `diag_3_group`

In [76]:
def classify_icd9(code):
    if code == 'Not_Documented':
        return 'Not_Documented'
    
    code = str(code).strip()
    
    # V and E codes
    if code.startswith('V'):
        return 'Supplementary factors (V-code)'
    
    if code.startswith('E'):
        return 'External causes (E-code)'
    
    # Numeric ICD-9 code
    try:
        value = float(code)
    except ValueError:
        return 'Other/Unknown'
    
    if 1 <= value <= 139:
        return 'Infectious & parasitic diseases'
    elif 140 <= value <= 239:
        return 'Neoplasms'
    elif 240 <= value <= 279:
        return 'Endocrine, nutritional & metabolic diseases'
    elif 280 <= value <= 289:
        return 'Blood diseases'
    elif 290 <= value <= 319:
        return 'Mental disorders'
    elif 320 <= value <= 389:
        return 'Nervous system & sense organs'
    elif 390 <= value <= 459:
        return 'Circulatory diseases'
    elif 460 <= value <= 519:
        return 'Respiratory diseases'
    elif 520 <= value <= 579:
        return 'Digestive diseases'
    elif 580 <= value <= 629:
        return 'Genitourinary diseases'
    elif 630 <= value <= 679:
        return 'Pregnancy & childbirth'
    elif 680 <= value <= 709:
        return 'Skin & subcutaneous tissue'
    elif 710 <= value <= 739:
        return 'Musculoskeletal system'
    elif 740 <= value <= 759:
        return 'Congenital anomalies'
    elif 760 <= value <= 779:
        return 'Perinatal conditions'
    elif 780 <= value <= 799:
        return 'Symptoms/signs/ill-defined conditions'
    elif 800 <= value <= 999:
        return 'Injury & poisoning'
    else:
        return 'Other/Unknown'

In [77]:
for col in diagnosis_cols:
    df_clean[f'{col}_group'] = df_clean[col].apply(classify_icd9)

In [78]:
for col in diagnosis_cols:
    group_col = f'{col}_group'
    
    print(f"\n{'=' * 60}")
    print(group_col)
    print(f"{'=' * 60}")
    print(df_clean[group_col].value_counts())


diag_1_group
diag_1_group
Circulatory diseases                           30336
Endocrine, nutritional & metabolic diseases    11459
Respiratory diseases                           10407
Digestive diseases                              9208
Symptoms/signs/ill-defined conditions           7636
Injury & poisoning                              6974
Genitourinary diseases                          5078
Musculoskeletal system                          4957
Neoplasms                                       3433
Infectious & parasitic diseases                 2768
Skin & subcutaneous tissue                      2530
Mental disorders                                2262
Supplementary factors (V-code)                  1644
Nervous system & sense organs                   1211
Blood diseases                                  1103
Pregnancy & childbirth                           687
Congenital anomalies                              51
Not_Documented                                    21
External causes (E-

In [79]:
for col in diagnosis_cols:
    group_col = f'{col}_group'
    
    print(
        col,
        "| original rows:", df_clean[col].notna().sum(),
        "| grouped rows:", df_clean[group_col].notna().sum()
    )

diag_1 | original rows: 101766 | grouped rows: 101766
diag_2 | original rows: 101766 | grouped rows: 101766
diag_3 | original rows: 101766 | grouped rows: 101766


## 2.26A Medication State Consistency

The medication variables describe treatment status using categorical states such as:

- `No`
- `Steady`
- `Up`
- `Down`

These states will be treated as categorical rather than assigning arbitrary numerical values.

Before further transformation, we will verify that the medication variables use a consistent set of states.

No medication values will be changed during this audit.

In [82]:
medication_cols = [
    col for col in medication_cols
    if col in df_clean.columns
]

print("Medication columns:", len(medication_cols))
print(medication_cols)

Medication columns: 21
['metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']


In [83]:
medication_states = {}

for col in medication_cols:
    medication_states[col] = sorted(
        df_clean[col].dropna().unique().tolist()
    )

medication_states

{'metformin': ['Down', 'No', 'Steady', 'Up'],
 'repaglinide': ['Down', 'No', 'Steady', 'Up'],
 'nateglinide': ['Down', 'No', 'Steady', 'Up'],
 'chlorpropamide': ['Down', 'No', 'Steady', 'Up'],
 'glimepiride': ['Down', 'No', 'Steady', 'Up'],
 'acetohexamide': ['No', 'Steady'],
 'glipizide': ['Down', 'No', 'Steady', 'Up'],
 'glyburide': ['Down', 'No', 'Steady', 'Up'],
 'tolbutamide': ['No', 'Steady'],
 'pioglitazone': ['Down', 'No', 'Steady', 'Up'],
 'rosiglitazone': ['Down', 'No', 'Steady', 'Up'],
 'acarbose': ['Down', 'No', 'Steady', 'Up'],
 'miglitol': ['Down', 'No', 'Steady', 'Up'],
 'troglitazone': ['No', 'Steady'],
 'tolazamide': ['No', 'Steady', 'Up'],
 'insulin': ['Down', 'No', 'Steady', 'Up'],
 'glyburide-metformin': ['Down', 'No', 'Steady', 'Up'],
 'glipizide-metformin': ['No', 'Steady'],
 'glimepiride-pioglitazone': ['No', 'Steady'],
 'metformin-rosiglitazone': ['No', 'Steady'],
 'metformin-pioglitazone': ['No', 'Steady']}

In [84]:
all_medication_states = sorted(
    set(
        state
        for states in medication_states.values()
        for state in states
    )
)

print("All medication states:")
print(all_medication_states)

All medication states:
['Down', 'No', 'Steady', 'Up']


## 2.26B Medication Summary Transformation

The 21 medication variables use four standardized treatment states:

- `No`
- `Steady`
- `Up`
- `Down`

The original medication variables will be retained.

Additional aggregate features will be created to summarize treatment status across the medication variables:

- `num_medications_active` — medications with a state other than `No`
- `num_medications_up` — medications whose treatment was increased
- `num_medications_down` — medications whose treatment was decreased
- `num_medications_steady` — medications maintained at a steady state

These features summarize medication treatment patterns without replacing the original medication variables.

In [86]:
df_clean['num_medications_active'] = (
    df_clean[medication_cols] != 'No'
).sum(axis=1)

df_clean['num_medications_up'] = (
    df_clean[medication_cols] == 'Up'
).sum(axis=1)

df_clean['num_medications_down'] = (
    df_clean[medication_cols] == 'Down'
).sum(axis=1)

df_clean['num_medications_steady'] = (
    df_clean[medication_cols] == 'Steady'
).sum(axis=1)

In [87]:
medication_summary = df_clean[
    [
        'num_medications_active',
        'num_medications_up',
        'num_medications_down',
        'num_medications_steady'
    ]
].describe()

medication_summary

,num_medications_active,num_medications_up,num_medications_down,num_medications_steady
count,101766.000000,101766.000000,101766.000000,101766.000000
mean,1.179706,0.146070,0.141373,0.892263
std,0.920190,0.370902,0.356356,0.897075
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,1.000000
75%,2.000000,0.000000,0.000000,1.000000
max,6.000000,3.000000,3.000000,6.000000


In [88]:
state_check = (
    df_clean['num_medications_active']
    ==
    (
        df_clean['num_medications_up']
        + df_clean['num_medications_down']
        + df_clean['num_medications_steady']
    )
)

print("Rows where medication counts do not reconcile:", (~state_check).sum())

Rows where medication counts do not reconcile: 0


## 2.26C Binary Medication Indicators

The dataset contains two binary medication-management variables:

- `change` — indicates whether a diabetes medication regimen was changed.
- `diabetesMed` — indicates whether diabetes medication was prescribed.

The original categorical variables will be preserved.

Explicit binary indicators will be created for easier analysis and later modeling:

- `medication_changed`
- `diabetes_medication_used`

`Yes/Ch` and `Yes/No` will be mapped to 1/0 respectively.

In [89]:
df_clean['medication_changed'] = (
    df_clean['change'] == 'Ch'
).astype(int)

df_clean['diabetes_medication_used'] = (
    df_clean['diabetesMed'] == 'Yes'
).astype(int)

In [90]:
print("Medication changed:")
print(
    pd.crosstab(
        df_clean['change'],
        df_clean['medication_changed']
    )
)

print("\nDiabetes medication used:")
print(
    pd.crosstab(
        df_clean['diabetesMed'],
        df_clean['diabetes_medication_used']
    )
)

Medication changed:
medication_changed      0      1
change                          
Ch                      0  47011
No                  54755      0

Diabetes medication used:
diabetes_medication_used      0      1
diabetesMed                           
No                        23403      0
Yes                           0  78363


## 2.27A Weight Documentation Indicator

The `weight` variable is missing for 96.86% of encounters.

Because the available values are categorical weight bands rather than exact measurements, imputing a numerical weight would introduce unsupported assumptions.

The original `weight` variable will therefore be preserved.

A new binary variable, `weight_documented`, will indicate whether a weight category was recorded:

- 1 → weight documented
- 0 → weight not documented

No weight values will be imputed.

In [91]:
df_clean['weight_documented'] = (df_clean['weight'].notna()).astype(int)

In [92]:
print(
    df_clean['weight_documented']
    .value_counts()
    .sort_index()
)

print("\nPercent:")
print(
    df_clean['weight_documented']
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

weight_documented
0    98569
1     3197
Name: count, dtype: int64

Percent:
weight_documented
0    96.86
1     3.14
Name: proportion, dtype: float64


## 2.28A Transform Glucose and A1C Categories

`max_glu_serum` and `A1Cresult` contain clinically ordered categories as well as `Not_Documented`.

The original categorical variables will be preserved.

For each variable, two additional features will be created:

1. A documentation indicator showing whether the measurement was documented.
2. An ordinal severity score for documented measurements.

`Not_Documented` will not be treated as a low clinical measurement.

In [93]:
# Maximum glucose serum

glucose_level_map = {
    'Norm': 0,
    '>200': 1,
    '>300': 2
}

df_clean['max_glu_serum_level'] = (
    df_clean['max_glu_serum']
    .map(glucose_level_map)
)

df_clean['max_glu_serum_documented'] = (
    df_clean['max_glu_serum'] != 'Not_Documented'
).astype(int)


# A1C

a1c_level_map = {
    'Norm': 0,
    '>7': 1,
    '>8': 2
}

df_clean['A1C_level'] = (
    df_clean['A1Cresult']
    .map(a1c_level_map)
)

df_clean['A1C_documented'] = (
    df_clean['A1Cresult'] != 'Not_Documented'
).astype(int)

In [94]:
print("Glucose level mapping:")
print(
    pd.crosstab(
        df_clean['max_glu_serum'],
        df_clean['max_glu_serum_level'],
        dropna=False
    )
)

print("\nGlucose documentation:")
print(
    pd.crosstab(
        df_clean['max_glu_serum'],
        df_clean['max_glu_serum_documented'],
        dropna=False
    )
)

print("\nA1C level mapping:")
print(
    pd.crosstab(
        df_clean['A1Cresult'],
        df_clean['A1C_level'],
        dropna=False
    )
)

print("\nA1C documentation:")
print(
    pd.crosstab(
        df_clean['A1Cresult'],
        df_clean['A1C_documented'],
        dropna=False
    )
)

Glucose level mapping:
max_glu_serum_level   0.0   1.0   2.0    NaN
max_glu_serum                               
>200                    0  1485     0      0
>300                    0     0  1264      0
Norm                 2597     0     0      0
Not_Documented          0     0     0  96420

Glucose documentation:
max_glu_serum_documented      0     1
max_glu_serum                        
>200                          0  1485
>300                          0  1264
Norm                          0  2597
Not_Documented            96420     0

A1C level mapping:
A1C_level        0.0   1.0   2.0    NaN
A1Cresult                              
>7                 0  3812     0      0
>8                 0     0  8216      0
Norm            4990     0     0      0
Not_Documented     0     0     0  84748

A1C documentation:
A1C_documented      0     1
A1Cresult                  
>7                  0  3812
>8                  0  8216
Norm                0  4990
Not_Documented  84748     0


## 2.29 Text Consistency Audit

Categorical variables should use consistent text representations.

Leading or trailing whitespace can create visually identical but technically different categories.

All string columns will therefore be checked for leading or trailing whitespace.

No values will be modified until the audit identifies affected columns.

In [95]:
string_cols = df_clean.select_dtypes(
    include=['str']
).columns

whitespace_audit = {}

for col in string_cols:
    values = df_clean[col].dropna().astype(str)
    
    whitespace_audit[col] = (
        values.str.strip() != values
    ).sum()

whitespace_audit = pd.Series(
    whitespace_audit
).sort_values(ascending=False)

whitespace_audit

race                        0
gender                      0
weight                      0
payer_code                  0
medical_specialty           0
diag_1                      0
diag_2                      0
diag_3                      0
max_glu_serum               0
A1Cresult                   0
metformin                   0
repaglinide                 0
nateglinide                 0
chlorpropamide              0
glimepiride                 0
acetohexamide               0
glipizide                   0
glyburide                   0
tolbutamide                 0
pioglitazone                0
rosiglitazone               0
acarbose                    0
miglitol                    0
troglitazone                0
tolazamide                  0
insulin                     0
glyburide-metformin         0
glipizide-metformin         0
glimepiride-pioglitazone    0
metformin-rosiglitazone     0
metformin-pioglitazone      0
change                      0
diabetesMed                 0
readmitted

In [97]:
print("Columns containing leading/trailing whitespace:")

print(
    whitespace_audit[
        whitespace_audit > 0
    ]
)

Columns containing leading/trailing whitespace:
Series([], dtype: int64)


## 2.30 Transformation Review

The cleaning and transformation steps have produced several derived variables while preserving the original source variables.

Before proceeding to modeling-readiness decisions, the current schema will be reviewed to ensure:

- expected columns are present,
- no accidental columns were created,
- data types remain appropriate,
- row count remains unchanged,
- and missingness has not unexpectedly increased.

This is a validation checkpoint before the next phase.

In [98]:
schema_review = pd.DataFrame({
    'dtype': df_clean.dtypes.astype(str),
    'unique_values': df_clean.nunique(dropna=False),
    'missing_values': df_clean.isna().sum()
})

print("Dataset shape:", df_clean.shape)

schema_review

Dataset shape: (101766, 66)


,dtype,unique_values,missing_values
encounter_id,int64,101766,0
patient_nbr,int64,71518,0
race,str,6,0
gender,str,3,0
age,category,10,0
...,...,...,...
weight_documented,int64,2,0
max_glu_serum_level,float64,4,96420
max_glu_serum_documented,int64,2,0
A1C_level,float64,4,84748


In [99]:
print("Rows:", len(df_clean))

print(
    "\nExpected original rows:",
    101766
)

print(
    "\nRow count preserved:",
    len(df_clean) == 101766
)

Rows: 101766

Expected original rows: 101766

Row count preserved: True


In [100]:
print("\nColumns created during transformation:")

created_cols = [
    'admission_type',
    'discharge_disposition',
    'admission_source',
    'diag_1_group',
    'diag_2_group',
    'diag_3_group',
    'num_medications_active',
    'num_medications_up',
    'num_medications_down',
    'num_medications_steady',
    'medication_changed',
    'diabetes_medication_used',
    'weight_documented',
    'max_glu_serum_level',
    'max_glu_serum_documented',
    'A1C_level',
    'A1C_documented',
    'readmitted_30d'
]

print(created_cols)

print(
    "\nMissing created columns:",
    [col for col in created_cols if col not in df_clean.columns]
)


Columns created during transformation:
['admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'diag_3_group', 'num_medications_active', 'num_medications_up', 'num_medications_down', 'num_medications_steady', 'medication_changed', 'diabetes_medication_used', 'weight_documented', 'max_glu_serum_level', 'max_glu_serum_documented', 'A1C_level', 'A1C_documented', 'readmitted_30d']

Missing created columns: []


## 2.31 Transformation Checkpoint

The cleaning and transformation phase has been completed for the current dataset.

Key outcomes:

- Original row count preserved: 101,766
- No duplicate encounter records
- Missing placeholders standardized
- Administrative IDs mapped to interpretable categories
- Diagnosis codes grouped into broad ICD-9 clinical categories
- Medication states summarized without destroying the original variables
- Weight missingness explicitly represented without imputation
- Glucose and A1C documentation/severity represented separately
- Binary medication indicators created
- Target transformed into a 30-day readmission indicator
- No unexpected values or transformation failures identified

The transformed dataset is now ready for a modeling-readiness and leakage assessment.

No final feature deletion or encoding is performed yet.

In [101]:
df_clean[
    [
        'discharge_disposition',
        'readmitted',
        'readmitted_30d'
    ]
].groupby(
    'discharge_disposition',
    observed=True
).agg(
    encounters=('readmitted_30d', 'size'),
    readmission_30d=('readmitted_30d', 'sum')
).assign(
    readmission_rate=lambda x:
        x['readmission_30d'] / x['encounters'] * 100
).sort_values(
    'encounters',
    ascending=False
)

,encounters,readmission_30d,readmission_rate
discharge_disposition,,,
Discharged to home,60234,5602,9.300395
Discharged/transferred to SNF,13954,2046,14.662462
Discharged/transferred to home with home health service,12902,1638,12.695706
Not_Documented,3691,459,12.435654
Discharged/transferred to another short term hospital,2128,342,16.071429
Discharged/transferred to another rehab fac including rehab units of a hospital .,1993,552,27.696939
Expired,1642,0,0.000000
Discharged/transferred to another type of inpatient care institution,1184,247,20.861486
Not Mapped,989,92,9.302326


## 2.31B Identify Structurally Non-Readmittable Encounters

Some discharge dispositions represent death or end-of-life care.

For these encounters, a 30-day hospital readmission may not be a meaningful outcome because the patient may no longer be at risk of returning to the hospital.

These records will therefore be identified separately before deciding whether they should be excluded from the modeling population.

No rows will be removed at this stage.

In [102]:
non_readmittable_ids = [11, 13, 14, 19, 20, 21]

non_readmittable = (
    df_clean['discharge_disposition_id']
    .isin(non_readmittable_ids)
)

print("Potentially non-readmittable encounters:")
print(non_readmittable.sum())

print(
    "\nPercentage:",
    round(non_readmittable.mean() * 100, 2)
)

print("\nBy disposition:")
print(
    df_clean.loc[
        non_readmittable,
        [
            'discharge_disposition_id',
            'discharge_disposition',
            'readmitted',
            'readmitted_30d'
        ]
    ].groupby(
        [
            'discharge_disposition_id',
            'discharge_disposition'
        ],
        observed=True
    )
    .agg(
        encounters=('readmitted_30d', 'size'),
        readmission_30d=('readmitted_30d', 'sum')
    )
)

Potentially non-readmittable encounters:
2423

Percentage: 2.38

By disposition:
                                                                             encounters  \
discharge_disposition_id discharge_disposition                                            
11                       Expired                                                   1642   
13                       Hospice / home                                             399   
14                       Hospice / medical facility                                 372   
19                       Expired at home. Medicaid only, hospice.                     8   
20                       Expired in a medical facility. Medicaid only, h...           2   

                                                                             readmission_30d  
discharge_disposition_id discharge_disposition                                                
11                       Expired                                                           

## 2.31C Identify Death-Related Encounters

The analysis shows that hospice encounters can still have a recorded 30-day readmission and therefore should remain eligible for the prediction task.

However, encounters where the patient died during the encounter or was discharged after death cannot subsequently experience a 30-day readmission.

These death-related encounters will be flagged separately.

No rows will be deleted at this stage.

This flag will be used later when defining the final modeling population.

In [ ]:
death_disposition_ids = [11, 19, 20, 21]

df_clean['death_related_disposition'] = (
    df_clean['discharge_disposition_id']
    .isin(death_disposition_ids)
).astype(int)

In [104]:
print(
    df_clean['death_related_disposition'].value_counts()
)

print("\nPercentage:")
print(
    df_clean['death_related_disposition']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

death_related_disposition
0    100114
1      1652
Name: count, dtype: int64

Percentage:
death_related_disposition
0    98.38
1     1.62
Name: proportion, dtype: float64


## 2.31D Define the Modeling Population

The cleaned dataset contains 1,652 encounters where the discharge disposition indicates that the patient died.

These encounters cannot subsequently experience a 30-day hospital readmission.

Therefore:

- `df_clean` remains the complete cleaned dataset.
- `df_model` will contain only encounters eligible for the 30-day readmission prediction task.
- Death-related encounters will be excluded from `df_model`.
- No records will be deleted from the cleaned source dataset.

In [105]:
df_model = df_clean[
    df_clean['death_related_disposition'] == 0
].copy()

print("df_clean shape:", df_clean.shape)
print("df_model shape:", df_model.shape)

df_clean shape: (101766, 67)
df_model shape: (100114, 67)


In [106]:
print("\nTarget distribution — modeling population:")
print(
    df_model['readmitted_30d'].value_counts()
)

print("\nTarget distribution (%):")
print(
    df_model['readmitted_30d']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)


Target distribution — modeling population:
readmitted_30d
0    88757
1    11357
Name: count, dtype: int64

Target distribution (%):
readmitted_30d
0    88.66
1    11.34
Name: proportion, dtype: float64


In [107]:
print(
    "\nDeath-related encounters remaining in df_model:",
    df_model['death_related_disposition'].sum()
)


Death-related encounters remaining in df_model: 0


## 2.32 Leakage Audit — Feature Availability

The prediction is made at or just before hospital discharge.

Each candidate feature must therefore represent information available by that point.

We will first inventory the current variables and classify them conceptually into:

- identifiers
- demographic information
- admission information
- discharge information
- clinical measurements
- utilization/history
- medication information
- diagnosis information
- target/derived target variables

No columns will be removed during this initial audit.

In [108]:
for i, col in enumerate(df_model.columns, start=1):
    print(f"{i:02d}. {col}")

01. encounter_id
02. patient_nbr
03. race
04. gender
05. age
06. weight
07. admission_type_id
08. discharge_disposition_id
09. admission_source_id
10. time_in_hospital
11. payer_code
12. medical_specialty
13. num_lab_procedures
14. num_procedures
15. num_medications
16. number_outpatient
17. number_emergency
18. number_inpatient
19. diag_1
20. diag_2
21. diag_3
22. number_diagnoses
23. max_glu_serum
24. A1Cresult
25. metformin
26. repaglinide
27. nateglinide
28. chlorpropamide
29. glimepiride
30. acetohexamide
31. glipizide
32. glyburide
33. tolbutamide
34. pioglitazone
35. rosiglitazone
36. acarbose
37. miglitol
38. troglitazone
39. tolazamide
40. insulin
41. glyburide-metformin
42. glipizide-metformin
43. glimepiride-pioglitazone
44. metformin-rosiglitazone
45. metformin-pioglitazone
46. change
47. diabetesMed
48. readmitted
49. readmitted_30d
50. admission_type
51. discharge_disposition
52. admission_source
53. diag_1_group
54. diag_2_group
55. diag_3_group
56. num_medications_activ

## 2.32A Initial Leakage Classification

The prediction point for this project is defined as at or just before hospital discharge.

Therefore, variables are evaluated based on whether they would be available at that point.

### Excluded from predictors

- `encounter_id` — encounter identifier
- `patient_nbr` — patient identifier
- `readmitted` — original target
- `readmitted_30d` — binary modeling target
- `death_related_disposition` — derived from discharge disposition and used to define the eligible modeling population

### Potential predictors

Demographic, admission, utilization-history, diagnosis, laboratory, medication, and encounter-level variables are retained for further evaluation and encoding.

### Discharge disposition

`discharge_disposition` is retained as a candidate predictor because the project's prediction point is at or immediately before discharge.

This decision is specific to the current prediction-point definition. If the prediction point changes to an earlier stage of hospitalization, discharge disposition must be reconsidered as potential leakage.

No feature columns are removed during this checkpoint.

## 2.32B Patient-Level Modeling Audit

The dataset contains multiple encounters for some patients.

Because the prediction target is encounter-level but patients can contribute multiple encounters, train/test splitting must later consider patient identity.

We will quantify the number of encounters contributed by each patient before deciding the appropriate validation strategy.

No rows or columns will be removed at this stage.

In [109]:
patient_encounter_counts = (
    df_model
    .groupby('patient_nbr')
    .size()
    .sort_values(ascending=False)
)

print("Unique patients:", patient_encounter_counts.shape[0])

print(
    "\nPatients with multiple encounters:",
    (patient_encounter_counts > 1).sum()
)

print(
    "\nMaximum encounters for one patient:",
    patient_encounter_counts.max()
)

print("\nEncounter distribution per patient:")
print(
    patient_encounter_counts.describe()
)

Unique patients: 70439

Patients with multiple encounters: 16483

Maximum encounters for one patient: 40

Encounter distribution per patient:
count    70439.000000
mean         1.421287
std          1.088970
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         40.000000
dtype: float64


In [110]:
print("\nTop 20 patients by encounter count:")
print(
    patient_encounter_counts.head(20)
)


Top 20 patients by encounter count:
patient_nbr
88785891    40
43140906    28
23199021    23
1660293     23
88227540    23
84428613    22
23643405    22
92709351    21
90609804    20
29903877    20
89472402    20
23398488    20
88789707    20
37096866    20
88681950    19
97391007    19
3401055     18
91160280    18
91751121    18
24011577    18
dtype: int64


## 2.32C Patient-Level Target Audit

The dataset contains multiple encounters for many patients.

Before defining the train/test strategy, we will examine whether patients can have different target outcomes across their encounters.

This will determine whether the target remains encounter-level while patient identity is used only for grouping during model validation.

No rows or columns will be removed at this stage.

In [111]:
patient_target_summary = (
    df_model
    .groupby('patient_nbr')['readmitted_30d']
    .agg(
        encounters='size',
        unique_target='nunique',
        positive_encounters='sum'
    )
)

print("Patients with multiple encounters:")
print(
    (patient_target_summary['encounters'] > 1).sum()
)

print("\nPatients with both target classes:")
print(
    (
        (patient_target_summary['encounters'] > 1) &
        (patient_target_summary['unique_target'] > 1)
    ).sum()
)

print("\nPatients with only positive encounters:")
print(
    (
        (patient_target_summary['encounters'] > 1) &
        (patient_target_summary['positive_encounters'] ==
         patient_target_summary['encounters'])
    ).sum()
)

print("\nPatients with only negative encounters:")
print(
    (
        (patient_target_summary['encounters'] > 1) &
        (patient_target_summary['positive_encounters'] == 0)
    ).sum()
)

Patients with multiple encounters:
16483

Patients with both target classes:
6377

Patients with only positive encounters:
203

Patients with only negative encounters:
9903


In [112]:
mixed_patients = patient_target_summary[
    (patient_target_summary['encounters'] > 1) &
    (patient_target_summary['unique_target'] > 1)
]

print("\nMixed-target patient encounter distribution:")
print(
    mixed_patients['encounters'].describe()
)


Mixed-target patient encounter distribution:
count    6377.000000
mean        3.447859
std         2.222592
min         2.000000
25%         2.000000
50%         3.000000
75%         4.000000
max        40.000000
Name: encounters, dtype: float64


## 2.32D Patient-Level Validation Strategy

The dataset contains repeated encounters for the same patient.

Among patients with multiple encounters:

- 6,377 patients have both positive and negative target outcomes.
- Therefore, `readmitted_30d` remains an encounter-level target.
- Patient-level target aggregation will not be performed.

`patient_nbr` will not be used as a predictive feature.

However, it will be retained as a grouping variable for the future train/validation/test split so that encounters from the same patient do not appear across different partitions.

This reduces patient-level contamination and provides a more realistic estimate of model performance on previously unseen patients.

In [113]:
utilization_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

df_model[utilization_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
number_outpatient,100114.0,0.369429,1.264006,0.0,0.0,0.0,0.0,42.0
number_emergency,100114.0,0.198334,0.935537,0.0,0.0,0.0,0.0,76.0
number_inpatient,100114.0,0.632829,1.261833,0.0,0.0,0.0,1.0,21.0


In [114]:
df_model[utilization_cols].sum()

number_outpatient    36985
number_emergency     19856
number_inpatient     63355
dtype: int64

## 2.32E Utilization Feature Semantics

The dataset contains counts of prior outpatient, emergency, and inpatient visits.

These variables may provide valuable historical utilization information for predicting 30-day readmission.

Before using them as predictors, their temporal meaning must be verified.

The key question is:

> Do these counts represent utilization before the current encounter?

If they represent prior utilization, they are potentially valid predictors at the discharge prediction point.

No transformation or removal will be performed until their semantics are confirmed.

In [115]:
utilization_cols = [
    'number_outpatient',
    'number_emergency',
    'number_inpatient'
]

for col in utilization_cols:
    print(f"\n{col}")
    print(df_model[col].value_counts().sort_index().head(15))


number_outpatient
number_outpatient
0     83624
1      8418
2      3542
3      2017
4      1083
5       518
6       300
7       154
8        98
9        82
10       57
11       41
12       30
13       30
14       28
Name: count, dtype: int64

number_emergency
number_emergency
0     88925
1      7538
2      2005
3       713
4       371
5       190
6        93
7        73
8        50
9        33
10       34
11       23
12       10
13       12
14        3
Name: count, dtype: int64

number_inpatient
number_inpatient
0     66672
1     19163
2      7382
3      3321
4      1588
5       800
6       477
7       264
8       146
9       111
10       59
11       49
12       34
13       18
14       10
Name: count, dtype: int64


## 2.33 Medication Feature Representation

The dataset contains 21 diabetes medication variables.

Each medication records whether the medication was:

- No
- Steady
- Up
- Down

We have already created aggregate medication features describing overall medication activity.

The next decision is whether the individual medication variables should also be retained.

Because many medications are extremely rare in this dataset, we will examine their state distributions before deciding whether to retain, simplify, or remove them.

In [116]:
medication_summary = []

for col in medication_cols:
    counts = df_model[col].value_counts()

    medication_summary.append({
        "medication": col,
        "unique_states": df_model[col].nunique(),
        "no_pct": counts.get("No", 0) / len(df_model) * 100,
        "steady_pct": counts.get("Steady", 0) / len(df_model) * 100,
        "up_pct": counts.get("Up", 0) / len(df_model) * 100,
        "down_pct": counts.get("Down", 0) / len(df_model) * 100,
    })

medication_summary = (
    pd.DataFrame(medication_summary)
    .sort_values("no_pct")
)

medication_summary

,medication,unique_states,no_pct,steady_pct,up_pct,down_pct
15,insulin,4,46.626845,30.296462,11.078371,11.998322
0,metformin,4,80.124658,18.235212,1.065785,0.574345
6,glipizide,4,87.409353,11.271151,0.766127,0.553369
7,glyburide,4,89.445033,9.189524,0.805082,0.560361
9,pioglitazone,4,92.727291,6.926104,0.229738,0.116867
10,rosiglitazone,4,93.678207,6.063088,0.172803,0.085902
4,glimepiride,4,94.858861,4.624728,0.323631,0.192780
1,repaglinide,4,98.474739,1.371437,0.108876,0.044949
16,glyburide-metformin,4,99.302795,0.683221,0.007991,0.005993
2,nateglinide,4,99.305791,0.659248,0.023973,0.010987


## 2.33 Medication Feature Representation

The dataset contains 21 medication variables with states:

- No
- Steady
- Up
- Down

The distributions are highly imbalanced. Insulin and several major medications have meaningful variation, while many medications are extremely rare.

Decision:

- Retain all original medication variables in `df_clean`.
- Retain the derived aggregate medication features.
- Do not remove rare medications solely based on frequency.
- Do not one-hot encode the medication variables in the cleaning notebook.
- Encoding and feature selection will be handled during the modeling pipeline.

This preserves the original clinical information while avoiding premature feature selection during data cleaning.

In [117]:
id_cols = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

df_clean[id_cols + [
    'admission_type',
    'discharge_disposition',
    'admission_source'
]].head(10)

,admission_type_id,discharge_disposition_id,admission_source_id,admission_type,discharge_disposition,admission_source
0,6,25,1,Not_Documented,Not Mapped,Physician Referral
1,1,1,7,Emergency,Discharged to home,Emergency Room
2,1,1,7,Emergency,Discharged to home,Emergency Room
3,1,1,7,Emergency,Discharged to home,Emergency Room
4,1,1,7,Emergency,Discharged to home,Emergency Room
5,2,1,2,Urgent,Discharged to home,Clinic Referral
6,3,1,2,Elective,Discharged to home,Clinic Referral
7,1,1,7,Emergency,Discharged to home,Emergency Room
8,2,1,4,Urgent,Discharged to home,Transfer from a hospital
9,3,3,4,Elective,Discharged/transferred to SNF,Transfer from a hospital


## 2.34 Categorical ID Variables

The dataset contains three coded categorical variables:

- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

The numeric values are category codes rather than continuous measurements.

Human-readable categorical versions were created through lookup mappings:

- `admission_type`
- `discharge_disposition`
- `admission_source`

Decision:

- Retain the original ID columns in `df_clean` for traceability.
- Use the descriptive categorical columns as the preferred representation for modeling.
- Exclude the numeric ID versions from the future modeling feature set.

In [118]:
df_clean[['weight', 'weight_documented']].head(20)

,weight,weight_documented
0,NaN,0
1,NaN,0
2,NaN,0
3,NaN,0
4,NaN,0
5,NaN,0
6,NaN,0
7,NaN,0
8,NaN,0
9,NaN,0


In [119]:
df_clean.groupby('weight_documented')['weight'].value_counts(dropna=False)

weight_documented  weight   
0                  NaN          98569
1                  [75-100)      1336
                   [50-75)        897
                   [100-125)      625
                   [125-150)      145
                   [25-50)         97
                   [0-25)          48
                   [150-175)       35
                   [175-200)       11
                   >200             3
Name: count, dtype: int64

## 2.35 Weight Representation

The `weight` variable is missing for 98,569 of 101,766 encounters (96.86%).

Only 3,197 encounters (3.14%) contain a documented weight category.

Validation confirmed:

- `weight_documented = 0` always corresponds to missing `weight`.
- `weight_documented = 1` always corresponds to a documented weight.
- No inconsistent records were found.

Decision:

- Preserve the original `weight` variable in `df_clean`.
- Preserve `weight_documented` as an explicit indicator of whether weight was documented.
- Do not impute the missing weight values.
- Missing weight will be handled during the modeling preprocessing stage.

This avoids introducing an artificial weight value into a variable with extremely high missingness while retaining the information that weight was documented.

In [120]:
weight_check = pd.crosstab(
    df_clean['weight_documented'],
    df_clean['weight'].isna()
)

weight_check

weight,False,True
weight_documented,,
0,0,98569
1,3197,0


## 2.36 Payer and Medical Specialty Representation

`payer_code` and `medical_specialty` contain substantial proportions of undocumented values.

The undocumented values have been explicitly represented as `Not_Documented`.

Because documentation status may itself contain useful information, `Not_Documented` will be retained as a legitimate categorical level rather than being imputed with the most frequent category.

Decision:

- Retain `payer_code`.
- Retain `medical_specialty`.
- Retain `Not_Documented` as an explicit category.
- Do not perform statistical imputation.
- Categorical encoding will be performed later during the modeling pipeline.

In [124]:
categorical_check_cols = [
    'payer_code',
    'medical_specialty'

]

for col in categorical_check_cols:
    print("=" * 60)
    print(col)
    print("NaN :",df_clean[col].isna().sum())
    print("Not documented :",(df_clean[col] == 'Not_Documented').sum())
    print("Unique:", df_clean[col].nunique())

payer_code
NaN : 0
Not documented : 40256
Unique: 18
medical_specialty
NaN : 0
Not documented : 49949
Unique: 73


## 2.36 Payer and Medical Specialty — Validation

Validation confirmed:

- `payer_code` contains no remaining NaN values.
- `medical_specialty` contains no remaining NaN values.
- `Not_Documented` is retained as an explicit categorical level.
- No statistical imputation was performed.

Decision:

Both variables are considered clean and ready for downstream analysis/modeling.

## 2.37 Final Categorical Value Audit

Before completing the cleaning stage, all categorical variables are checked for:

- remaining NaN values
- unexpected values
- accidental empty strings
- unexpected whitespace

This validation ensures that the transformations performed during cleaning did not introduce inconsistent categorical values.

In [126]:
categorical_cols = df_clean.select_dtypes(
    include=['object', 'string', 'category']
).columns.tolist()

categorical_audit = []

for col in categorical_cols:
    categorical_audit.append({
        'column': col,
        'dtype': str(df_clean[col].dtype),
        'missing': df_clean[col].isna().sum(),
        'empty_strings': (df_clean[col] == '').sum(),
        'unique_values': df_clean[col].nunique()
    })

categorical_audit = pd.DataFrame(categorical_audit)

categorical_audit

,column,dtype,missing,empty_strings,unique_values
0,race,str,0,0,6
1,gender,str,0,0,3
2,age,category,0,0,10
3,weight,str,98569,0,9
4,admission_type_id,category,0,0,8
5,discharge_disposition_id,category,0,0,26
6,admission_source_id,category,0,0,17
7,payer_code,str,0,0,18
8,medical_specialty,str,0,0,73
9,diag_1,str,0,0,717


In [127]:
print("Categorical columns containing NaN:")
print(
    categorical_audit[
        categorical_audit['missing'] > 0
    ]
)

print("\nCategorical columns containing empty strings:")
print(
    categorical_audit[
        categorical_audit['empty_strings'] > 0
    ]
)

Categorical columns containing NaN:
   column dtype  missing  empty_strings  unique_values
3  weight   str    98569              0              9

Categorical columns containing empty strings:
Empty DataFrame
Columns: [column, dtype, missing, empty_strings, unique_values]
Index: []


## 2.37 Final Categorical Value Audit — Decision

The categorical audit found:

- No empty-string values.
- No unexpected missing values in the cleaned categorical variables.
- `weight` contains 98,569 missing values by design.
- The missingness in `weight` is explicitly represented by `weight_documented`.
- Original coded ID variables and their descriptive lookup columns are both retained in `df_clean` for traceability.

The missing values in `weight` are therefore considered intentional rather than a cleaning failure.

The coded ID columns will be excluded from the future modeling feature set in favor of their descriptive categorical representations.

## 2.38 Final Feature Inventory

Before constructing the modeling feature set, every column is classified according to its role:

1. Identifier
2. Target
3. Potential leakage
4. Original categorical variable
5. Transformed categorical variable
6. Numerical feature
7. Derived feature
8. Documentation/missingness indicator

This inventory will be used to define the final modeling dataset without accidentally including identifiers or target-related information.

In [129]:
for i, col in enumerate(df_model.columns, start=1):
    print(f"{i:02d}. {col}")

01. encounter_id
02. patient_nbr
03. race
04. gender
05. age
06. weight
07. admission_type_id
08. discharge_disposition_id
09. admission_source_id
10. time_in_hospital
11. payer_code
12. medical_specialty
13. num_lab_procedures
14. num_procedures
15. num_medications
16. number_outpatient
17. number_emergency
18. number_inpatient
19. diag_1
20. diag_2
21. diag_3
22. number_diagnoses
23. max_glu_serum
24. A1Cresult
25. metformin
26. repaglinide
27. nateglinide
28. chlorpropamide
29. glimepiride
30. acetohexamide
31. glipizide
32. glyburide
33. tolbutamide
34. pioglitazone
35. rosiglitazone
36. acarbose
37. miglitol
38. troglitazone
39. tolazamide
40. insulin
41. glyburide-metformin
42. glipizide-metformin
43. glimepiride-pioglitazone
44. metformin-rosiglitazone
45. metformin-pioglitazone
46. change
47. diabetesMed
48. readmitted
49. readmitted_30d
50. admission_type
51. discharge_disposition
52. admission_source
53. diag_1_group
54. diag_2_group
55. diag_3_group
56. num_medications_activ

In [130]:
print("df_clean shape:", df_clean.shape)
print("df_model shape:", df_model.shape)

df_clean shape: (101766, 67)
df_model shape: (100114, 67)


## 2.38 Feature Role Classification

Each variable is classified according to its role in the downstream machine-learning workflow.

The classification distinguishes:

- identifiers
- target variables
- eligibility/leakage-related variables
- categorical identifiers
- original clinical variables
- transformed clinical variables
- derived features

Columns are retained in `df_clean` unless there is a specific reason to remove them.

The final modeling feature set will be created separately after the leakage audit.

In [131]:
feature_roles = {
    'identifier': [
        'encounter_id',
        'patient_nbr'
    ],

    'target': [
        'readmitted',
        'readmitted_30d'
    ],

    'eligibility_or_leakage': [
        'death_related_disposition'
    ],

    'categorical_id': [
        'admission_type_id',
        'discharge_disposition_id',
        'admission_source_id'
    ]
}

role_rows = []

for role, columns in feature_roles.items():
    for col in columns:
        role_rows.append({
            'column': col,
            'role': role,
            'exists': col in df_model.columns
        })

feature_role_audit = pd.DataFrame(role_rows)

feature_role_audit

,column,role,exists
0,encounter_id,identifier,True
1,patient_nbr,identifier,True
2,readmitted,target,True
3,readmitted_30d,target,True
4,death_related_disposition,eligibility_or_leakage,True
5,admission_type_id,categorical_id,True
6,discharge_disposition_id,categorical_id,True
7,admission_source_id,categorical_id,True


In [132]:
print("Columns missing from df_model:")
print(
    feature_role_audit.loc[
        ~feature_role_audit['exists'],
        'column'
    ].tolist()
)

Columns missing from df_model:
[]


## 2.39 Prediction Point and Leakage Definition

### Prediction point

The model is designed to estimate 30-day readmission risk at or just before hospital discharge.

Therefore, a feature is considered potentially valid if its information would reasonably be available by the time of discharge.

### Valid candidate features

Features describing the patient's:

- demographics
- diagnoses
- prior healthcare utilization
- laboratory measurements
- procedures
- medication status
- medication changes
- hospital stay characteristics
- discharge disposition

may be used as candidate predictors if they are available by the prediction point.

### Excluded variables

The following variables are excluded from prediction:

- `readmitted`
- `readmitted_30d`
- `death_related_disposition`
- `encounter_id`
- `patient_nbr`

The coded ID versions of categorical variables will also be excluded in favor of their descriptive representations.

The final feature set will be finalized after the remaining leakage and feature-quality audits.

In [133]:
candidate_groups = {
    'demographics': [
        'race',
        'gender',
        'age',
        'weight',
        'weight_documented'
    ],

    'admission_discharge': [
        'admission_type',
        'discharge_disposition',
        'admission_source'
    ],

    'hospital_course': [
        'time_in_hospital',
        'num_lab_procedures',
        'num_procedures',
        'num_medications',
        'number_diagnoses'
    ],

    'prior_utilization': [
        'number_outpatient',
        'number_emergency',
        'number_inpatient'
    ],

    'diagnoses': [
        'diag_1',
        'diag_2',
        'diag_3',
        'diag_1_group',
        'diag_2_group',
        'diag_3_group'
    ],

    'glycemic_measurements': [
        'max_glu_serum',
        'max_glu_serum_level',
        'max_glu_serum_documented',
        'A1Cresult',
        'A1C_level',
        'A1C_documented'
    ],

    'medications': medication_cols,

    'medication_aggregates': [
        'num_medications_active',
        'num_medications_up',
        'num_medications_down',
        'num_medications_steady',
        'medication_changed',
        'diabetes_medication_used'
    ],

    'administrative': [
        'payer_code',
        'medical_specialty'
    ]
}

candidate_rows = []

for group, columns in candidate_groups.items():
    for col in columns:
        candidate_rows.append({
            'group': group,
            'column': col,
            'exists': col in df_model.columns
        })

candidate_audit = pd.DataFrame(candidate_rows)

candidate_audit

,group,column,exists
0,demographics,race,True
1,demographics,gender,True
2,demographics,age,True
3,demographics,weight,True
4,demographics,weight_documented,True
5,admission_discharge,admission_type,True
6,admission_discharge,discharge_disposition,True
7,admission_discharge,admission_source,True
8,hospital_course,time_in_hospital,True
9,hospital_course,num_lab_procedures,True


In [134]:
print("Candidate columns missing from df_model:")
print(
    candidate_audit.loc[
        ~candidate_audit['exists'],
        'column'
    ].tolist()
)

Candidate columns missing from df_model:
[]


Step 2.41 — Create the provisional ML feature list

In [135]:
exclude_from_model = [
    'encounter_id',
    'patient_nbr',
    'readmitted',
    'readmitted_30d',
    'death_related_disposition',

    # Numeric coded versions
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id'
]

provisional_features = [
    col
    for col in df_model.columns
    if col not in exclude_from_model
]

print("Total columns in df_model:", df_model.shape[1])
print("Excluded columns:", len(exclude_from_model))
print("Provisional model features:", len(provisional_features))

print("\nExcluded:")
print(exclude_from_model)

print("\nProvisional features:")
for i, col in enumerate(provisional_features, 1):
    print(f"{i:02d}. {col}")

Total columns in df_model: 67
Excluded columns: 8
Provisional model features: 59

Excluded:
['encounter_id', 'patient_nbr', 'readmitted', 'readmitted_30d', 'death_related_disposition', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id']

Provisional features:
01. race
02. gender
03. age
04. weight
05. time_in_hospital
06. payer_code
07. medical_specialty
08. num_lab_procedures
09. num_procedures
10. num_medications
11. number_outpatient
12. number_emergency
13. number_inpatient
14. diag_1
15. diag_2
16. diag_3
17. number_diagnoses
18. max_glu_serum
19. A1Cresult
20. metformin
21. repaglinide
22. nateglinide
23. chlorpropamide
24. glimepiride
25. acetohexamide
26. glipizide
27. glyburide
28. tolbutamide
29. pioglitazone
30. rosiglitazone
31. acarbose
32. miglitol
33. troglitazone
34. tolazamide
35. insulin
36. glyburide-metformin
37. glipizide-metformin
38. glimepiride-pioglitazone
39. metformin-rosiglitazone
40. metformin-pioglitazone
41. change
42. diabetesMed
43. 

In [136]:
feature_dtype_audit = (
    df_model[provisional_features]
    .dtypes
    .astype(str)
    .value_counts()
)

print("Provisional feature dtypes:")
print(feature_dtype_audit)

Provisional feature dtypes:
str         36
int64       17
category     4
float64      2
Name: count, dtype: int64


In [137]:
print("\nDetailed feature dtype audit:")
print(
    pd.DataFrame({
        'column': provisional_features,
        'dtype': df_model[provisional_features].dtypes.astype(str).values
    })
)


Detailed feature dtype audit:
                      column     dtype
0                       race       str
1                     gender       str
2                        age  category
3                     weight       str
4           time_in_hospital     int64
5                 payer_code       str
6          medical_specialty       str
7         num_lab_procedures     int64
8             num_procedures     int64
9            num_medications     int64
10         number_outpatient     int64
11          number_emergency     int64
12          number_inpatient     int64
13                    diag_1       str
14                    diag_2       str
15                    diag_3       str
16          number_diagnoses     int64
17             max_glu_serum       str
18                 A1Cresult       str
19                 metformin       str
20               repaglinide       str
21               nateglinide       str
22            chlorpropamide       str
23               glimepiride     

In [138]:
print("\nCategorical features:")
print(
    df_model[provisional_features]
    .select_dtypes(include=['object', 'string', 'category'])
    .columns
    .tolist()
)

print("\nNumeric features:")
print(
    df_model[provisional_features]
    .select_dtypes(include=['number'])
    .columns
    .tolist()
)


Categorical features:
['race', 'gender', 'age', 'weight', 'payer_code', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'max_glu_serum', 'A1Cresult', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'admission_type', 'discharge_disposition', 'admission_source', 'diag_1_group', 'diag_2_group', 'diag_3_group']

Numeric features:
['time_in_hospital', 'num_lab_procedures', 'num_procedures', 'num_medications', 'number_outpatient', 'number_emergency', 'number_inpatient', 'number_diagnoses', 'num_medications_active', 'num_medications_up', 'num_medications_down', 'num_medications_steady', 'medication_changed', 'diabetes_medication_used', 'weight_documented', 'm

In [139]:
missing_audit = pd.DataFrame({
    'dtype': df_model[provisional_features].dtypes.astype(str),
    'missing_count': df_model[provisional_features].isna().sum(),
    'missing_pct': (
        df_model[provisional_features].isna().mean() * 100
    ).round(2)
})

missing_audit = (
    missing_audit
    .sort_values('missing_count', ascending=False)
)

missing_audit

,dtype,missing_count,missing_pct
weight,str,96958,96.85
max_glu_serum_level,float64,94890,94.78
A1C_level,float64,83238,83.14
age,category,0,0.00
time_in_hospital,int64,0,0.00
payer_code,str,0,0.00
medical_specialty,str,0,0.00
num_lab_procedures,int64,0,0.00
num_procedures,int64,0,0.00
gender,str,0,0.00


In [140]:
print("Features with missing values:")
print(
    missing_audit[
        missing_audit['missing_count'] > 0
    ]
)

Features with missing values:
                       dtype  missing_count  missing_pct
weight                   str          96958        96.85
max_glu_serum_level  float64          94890        94.78
A1C_level            float64          83238        83.14


In [141]:
print("=" * 60)
print("FINAL DATASET INTEGRITY CHECK")
print("=" * 60)

# ----------------------------------------------------------
# 1. Shape
# ----------------------------------------------------------
print("\n1. DATASET SHAPE")
print("df_clean:", df_clean.shape)
print("df_model:", df_model.shape)

# ----------------------------------------------------------
# 2. Target
# ----------------------------------------------------------
print("\n2. TARGET")
print("Target column:", 'readmitted_30d')
print("Missing target:", df_model['readmitted_30d'].isna().sum())
print("\nTarget distribution:")
print(df_model['readmitted_30d'].value_counts())

print("\nTarget distribution (%):")
print(
    df_model['readmitted_30d']
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

# ----------------------------------------------------------
# 3. Encounter integrity
# ----------------------------------------------------------
print("\n3. ENCOUNTER INTEGRITY")
print("Unique encounters:", df_model['encounter_id'].nunique())
print("Duplicate encounter IDs:", df_model['encounter_id'].duplicated().sum())

# ----------------------------------------------------------
# 4. Patient integrity
# ----------------------------------------------------------
print("\n4. PATIENT INTEGRITY")
print("Unique patients:", df_model['patient_nbr'].nunique())

patient_counts = df_model['patient_nbr'].value_counts()

print(
    "Patients with multiple encounters:",
    (patient_counts > 1).sum()
)

print(
    "Maximum encounters for one patient:",
    patient_counts.max()
)

# ----------------------------------------------------------
# 5. Duplicate rows
# ----------------------------------------------------------
print("\n5. DUPLICATE ROWS")
print(
    "Completely duplicated rows:",
    df_model.duplicated().sum()
)

# ----------------------------------------------------------
# 6. Death-related encounters
# ----------------------------------------------------------
print("\n6. ELIGIBILITY CHECK")
print(
    "Death-related encounters remaining:",
    df_model['death_related_disposition'].sum()
)

# ----------------------------------------------------------
# 7. Missing values
# ----------------------------------------------------------
print("\n7. MISSING VALUES")

missing_features = (
    df_model[provisional_features]
    .isna()
    .sum()
)

print(
    missing_features[
        missing_features > 0
    ]
)

# ----------------------------------------------------------
# 8. Feature count
# ----------------------------------------------------------
print("\n8. FEATURE COUNT")
print("Total df_model columns:", df_model.shape[1])
print("Excluded columns:", len(exclude_from_model))
print("Provisional model features:", len(provisional_features))

# ----------------------------------------------------------
# 9. Required columns
# ----------------------------------------------------------
print("\n9. REQUIRED COLUMNS")

required_columns = [
    'patient_nbr',
    'readmitted_30d',
    'weight_documented',
    'max_glu_serum_documented',
    'A1C_documented',
    'medication_changed',
    'diabetes_medication_used',
    'diag_1_group',
    'diag_2_group',
    'diag_3_group'
]

missing_required = [
    col
    for col in required_columns
    if col not in df_model.columns
]

print("Missing required columns:", missing_required)

# ----------------------------------------------------------
# 10. Final status
# ----------------------------------------------------------
print("\n" + "=" * 60)
print("FINAL STATUS")
print("=" * 60)

checks = {
    "Correct modeling population": len(df_model) == 100114,
    "Target has no missing values": df_model['readmitted_30d'].isna().sum() == 0,
    "No duplicate encounter IDs": df_model['encounter_id'].duplicated().sum() == 0,
    "No duplicate rows": df_model.duplicated().sum() == 0,
    "No death-related encounters": df_model['death_related_disposition'].sum() == 0,
    "59 provisional features": len(provisional_features) == 59,
    "Required columns present": len(missing_required) == 0
}

for check, result in checks.items():
    print(f"{'PASS' if result else 'FAIL'} — {check}")

FINAL DATASET INTEGRITY CHECK

1. DATASET SHAPE
df_clean: (101766, 67)
df_model: (100114, 67)

2. TARGET
Target column: readmitted_30d
Missing target: 0

Target distribution:
readmitted_30d
0    88757
1    11357
Name: count, dtype: int64

Target distribution (%):
readmitted_30d
0    88.66
1    11.34
Name: proportion, dtype: float64

3. ENCOUNTER INTEGRITY
Unique encounters: 100114
Duplicate encounter IDs: 0

4. PATIENT INTEGRITY
Unique patients: 70439
Patients with multiple encounters: 16483
Maximum encounters for one patient: 40

5. DUPLICATE ROWS
Completely duplicated rows: 0

6. ELIGIBILITY CHECK
Death-related encounters remaining: 0

7. MISSING VALUES
weight                 96958
max_glu_serum_level    94890
A1C_level              83238
dtype: int64

8. FEATURE COUNT
Total df_model columns: 67
Excluded columns: 8
Provisional model features: 59

9. REQUIRED COLUMNS
Missing required columns: []

FINAL STATUS
PASS — Correct modeling population
PASS — Target has no missing values
PASS 

# Final Cleaning & Transformation Summary

## Dataset Status

The dataset cleaning and transformation process has been completed and validated.

### Dataset populations

| Dataset | Encounters |
|---|---:|
| `df_clean` | 101,766 |
| `df_model` | 100,114 |
| Excluded death-related encounters | 1,652 |

The full transformed dataset is preserved in `df_clean`.

`df_model` represents the eligible modeling population after excluding encounters associated with death-related discharge dispositions because a 30-day readmission outcome is not meaningful for these encounters.

---

## Target Definition

The original `readmitted` variable contained three categories:

- `<30` — readmitted within 30 days
- `>30` — readmitted after 30 days
- `NO` — not readmitted

For binary classification:

- `<30` → `1`
- `>30` → `0`
- `NO` → `0`

The final target is:

`readmitted_30d`

### Modeling target distribution

- Negative: 88.66%
- Positive: 11.34%

The target contains no missing values.

---

## Transformations Performed

### Administrative code lookups

Numeric identifiers were supplemented with descriptive categorical variables:

- `admission_type`
- `discharge_disposition`
- `admission_source`

The original ID columns are retained for traceability but excluded from the provisional model feature set.

### Diagnosis transformation

The three diagnosis fields were classified into clinically meaningful ICD-based groups:

- `diag_1_group`
- `diag_2_group`
- `diag_3_group`

The original diagnosis codes were retained.

### Medication transformation

The 21 medication variables were summarized into:

- `num_medications_active`
- `num_medications_up`
- `num_medications_down`
- `num_medications_steady`
- `medication_changed`
- `diabetes_medication_used`

The original medication variables were retained.

### Clinical measurement transformation

Glucose and A1C results were converted into numeric representations while preserving whether the measurement was documented:

- `max_glu_serum_level`
- `max_glu_serum_documented`
- `A1C_level`
- `A1C_documented`

### Weight documentation

A binary indicator was created:

- `weight_documented`

The original weight variable was retained.

### Demographic standardization

The `gender` field was standardized so that the previously invalid/unknown category is represented consistently as `Not_Documented`.

The `age` variable was converted to an ordered categorical variable using the original age bands.

---

## Missing Data Strategy

No arbitrary imputation was performed during cleaning.

Remaining missing values are limited to:

- `weight`
- `max_glu_serum_level`
- `A1C_level`

These correspond to substantial non-documentation in the original data.

Documentation indicators were created to preserve this information.

Any model-specific imputation will be performed later inside the modeling preprocessing pipeline using training data only.

---

## Modeling Feature Boundary

The provisional modeling feature set contains 59 predictors.

The following variables are excluded from the provisional model feature set:

- `encounter_id`
- `patient_nbr`
- `readmitted`
- `readmitted_30d`
- `death_related_disposition`
- `admission_type_id`
- `discharge_disposition_id`
- `admission_source_id`

Patient and encounter identifiers are excluded because they are identifiers rather than meaningful predictive features.

The original numeric administrative IDs are excluded in favor of their descriptive categorical representations.

The original target and derived target are excluded to prevent target leakage.

---

## Data Integrity Validation

Final validation confirmed:

- 100,114 eligible modeling encounters
- 100,114 unique encounter IDs
- 0 duplicate encounter IDs
- 0 completely duplicated rows
- 70,439 unique patients
- 16,483 patients with multiple encounters
- Maximum of 40 encounters for a single patient
- 0 missing target values
- 0 death-related encounters remaining in the modeling population
- 59 provisional modeling predictors
- All required engineered columns present

## Status

**Cleaning and transformation complete.**

The validated `df_model` dataset is ready for the next stage:

**modeling preparation, patient-level train/validation/test splitting, categorical encoding, missing-value handling, class-imbalance strategy, and baseline modeling.**